In [ ]:
# BLOCK 1 (UPDATED): Environment - pinned to the working Kaggle versions (Python 3.13)
!pip install -q "opacus==1.6.0" "imbalanced-learn==0.14.2"

import importlib, json, sys
versions = {"python": sys.version.split()[0]}
for pkg in ["numpy", "sklearn", "imblearn", "torch", "pandas", "opacus"]:
    versions[pkg] = importlib.import_module(pkg).__version__
print(json.dumps(versions, indent=2))
json.dump(versions, open("env_versions.json", "w"), indent=2)

assert versions["numpy"].startswith("2.1"), "numpy version changed - check env"
assert versions["sklearn"].startswith("1.6"), "sklearn version changed - check env"

In [ ]:
# BLOCK 2 (UPDATED): Imports - only what the pipeline actually uses
import os
import random
import warnings
from collections import deque

# Data manipulation
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Torch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset

# Sklearn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, auc)

# Imbalanced-learn (what the resampling block actually uses)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Warnings stay ON
warnings.simplefilter("default")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

In [ ]:
# BLOCK 3 (UPDATED): Device + reproducibility
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import glob
print(glob.glob("/kaggle/input/**/*.csv", recursive=True))

In [ ]:
# BLOCK 4 (UPDATED): Load and explore the dataset
file_path = "/kaggle/input/datasets/fadiabuzwayed/ton-iot-train-test-network/TON_IoT_Train_Test_Network.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Dataset not found at: {file_path}")

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values (total):", int(df.isnull().sum().sum()))

# Class distribution - computed, not hardcoded
label_distribution = df['type'].value_counts()
print("\nClass distribution:\n", label_distribution)
normal_pct = 100 * (df['type'] == 'normal').mean()
print(f"\nNormal: {normal_pct:.1f}% | Attack: {100 - normal_pct:.1f}%")

plt.figure(figsize=(8, 5))
label_distribution.plot(kind='bar')
plt.title('Class Distribution (type)')
plt.xlabel('Class'); plt.ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# BLOCK 5 (UPDATED): Drop leak/shortcut columns, THEN remove duplicates
LEAK_COLUMNS     = ['label']                                          # binary copy of 'type'
SHORTCUT_COLUMNS = ['ts', 'src_ip', 'dst_ip', 'src_port', 'dst_port'] # identify the capture, not behaviour
df = df.drop(columns=LEAK_COLUMNS + SHORTCUT_COLUMNS)
assert 'label' not in df.columns

n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Exact duplicates removed: {n0 - len(df)}")

# Conflicting rows: identical features but different class -> unlearnable, remove
feat_cols = [c for c in df.columns if c != 'type']
conflicts = df.duplicated(subset=feat_cols, keep=False)
print(f"Conflicting rows removed: {int(conflicts.sum())}")
df = df[~conflicts].reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(df['type'].value_counts())

# Upload and explore the dataset

In [ ]:
# BLOCK 6 (UPDATED): Encode categorical columns + keep the TRUE class-name mapping
nominal = [c for c in df.columns if df[c].dtype == 'object' and c != 'type']
print("Categorical features:", nominal)
for col in nominal:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

type_encoder = LabelEncoder()
df['type'] = type_encoder.fit_transform(df['type'].astype(str))
class_names = list(type_encoder.classes_)          # use this everywhere for class names
print("Class mapping:", dict(enumerate(class_names)))

# Step 2: Segreggation *columns*

In [ ]:
# BLOCK 7 (UPDATED): Split features/target + leakage checks
target_column = 'type'
X = df.drop(columns=[target_column])
y = df[target_column]

# Leakage guards
for c in ['type', 'label', 'ts', 'src_ip', 'dst_ip', 'src_port', 'dst_port']:
    assert c not in X.columns, f"Leakage guard: '{c}' still in features"
assert X.shape[1] == 38, f"Expected 38 features, got {X.shape[1]}"
assert X.shape[0] == y.shape[0]
assert int(X.isnull().sum().sum()) == 0 and int(y.isnull().sum()) == 0, "Missing values found"

print(f"Features: {X.shape} | Target: {y.shape}")
print("Feature columns:", X.columns.tolist())

In [ ]:
# BLOCK 8 (UPDATED): Stratified 70/10/20 split + quantile transform fit on TRAIN only
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

idx = np.arange(len(X))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
val_idx, test_idx   = train_test_split(temp_idx, test_size=2/3, stratify=y.iloc[temp_idx], random_state=SEED)
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

# Quantile -> normal: handles heavy-tailed counts and label-encoded categoricals (fit on train only)
scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                             subsample=200_000, random_state=SEED).fit(X.iloc[train_idx])

X_train = scaler.transform(X.iloc[train_idx]).astype(np.float32); y_train = y.iloc[train_idx].to_numpy()
X_val   = scaler.transform(X.iloc[val_idx]).astype(np.float32);   y_val   = y.iloc[val_idx].to_numpy()
X_test  = scaler.transform(X.iloc[test_idx]).astype(np.float32);  y_test  = y.iloc[test_idx].to_numpy()

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long())
val_dataset   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val).long())
test_dataset  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test).long())

split_table = pd.DataFrame({
    'train': pd.Series(y_train).value_counts(),
    'val':   pd.Series(y_val).value_counts(),
    'test':  pd.Series(y_test).value_counts(),
}).sort_index()
split_table.index = [class_names[i] for i in split_table.index]
print(f"Train {len(y_train)} | Val {len(y_val)} | Test {len(y_test)}")
print(split_table)

In [ ]:
# BLOCK 8b (NEW): ExtraTrees feature selection - fitted on TRAIN only and actually applied
from sklearn.ensemble import ExtraTreesClassifier

TOP_K = 20
critical_security_features = [
    'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected',
    'ssl_cipher', 'ssl_established', 'ssl_issuer',
    'http_method', 'http_version', 'http_status_code',
    'http_request_body_len', 'http_resp_mime_types',
    'weird_name', 'weird_addl'
]

et = ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
importance = pd.Series(et.feature_importances_, index=X.columns).sort_values(ascending=False)

keep = set(importance.head(TOP_K).index) | {f for f in critical_security_features if f in X.columns}
selected_features = [c for c in X.columns if c in keep]
sel_idx = [X.columns.get_loc(c) for c in selected_features]

X_train, X_val, X_test = X_train[:, sel_idx], X_val[:, sel_idx], X_test[:, sel_idx]
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long())
val_dataset   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val).long())
test_dataset  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test).long())

print(f"Selected {len(selected_features)} features:", selected_features)
print("\nTop-20 importance:\n", importance.head(TOP_K))

In [ ]:
# BLOCK 9 (UPDATED): Hybrid over/under-sampling on TRAIN only (capped) + class weights
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

OVERSAMPLE_CAP = 10                                   # rare classes grow at most 10x
counts = Counter(y_train)
target = int(np.median(list(counts.values())))

over  = {c: min(target, n * OVERSAMPLE_CAP) for c, n in counts.items() if n < target}
under = {c: target for c, n in counts.items() if n > target}

X_res, y_res = RandomOverSampler(sampling_strategy=over, random_state=SEED).fit_resample(X_train, y_train)
X_res, y_res = RandomUnderSampler(sampling_strategy=under, random_state=SEED).fit_resample(X_res, y_res)

X_train_res, y_train_res = X_res.astype(np.float32), y_res.astype(np.int64)
balanced_train_ds = TensorDataset(torch.from_numpy(X_train_res), torch.from_numpy(y_train_res).long())

res_table = pd.DataFrame({'before': pd.Series(y_train).value_counts(),
                          'after':  pd.Series(y_train_res).value_counts()}).sort_index()
res_table.index = [class_names[i] for i in res_table.index]
print(f"Median target per class: {target}\n", res_table)

cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train_res)
class_weights = (np.sqrt(cw) / np.sqrt(cw).mean()).astype(np.float32)
for name, w in zip(class_names, class_weights):
    print(f"{name:12s} weight = {w:.3f}")

In [ ]:
# BLOCK 10 (UPDATED): Non-IID Dirichlet client split - seeded, no IID fallback
NUM_CLIENTS = 15
ALPHA = 2.0                     # Dirichlet concentration (lower = more non-IID)
MIN_PER_CLIENT = 100            # minimum total samples per client

def dirichlet_split(labels, num_clients, alpha, rng, min_per_client):
    num_classes = int(labels.max()) + 1
    for _ in range(100):
        client_idx = [[] for _ in range(num_clients)]
        for c in range(num_classes):
            idx_c = np.where(labels == c)[0]
            rng.shuffle(idx_c)
            p = rng.dirichlet([alpha] * num_clients)
            cuts = (np.cumsum(p) * len(idx_c)).astype(int)[:-1]
            for k, part in enumerate(np.split(idx_c, cuts)):
                client_idx[k].extend(part.tolist())
        if min(len(ci) for ci in client_idx) >= min_per_client:
            return [np.array(ci) for ci in client_idx]
    raise RuntimeError("Could not satisfy MIN_PER_CLIENT - lower it or raise ALPHA")

rng = np.random.default_rng(SEED)
client_indices = dirichlet_split(y_train_res, NUM_CLIENTS, ALPHA, rng, MIN_PER_CLIENT)

all_idx = np.concatenate(client_indices)
assert len(all_idx) == len(y_train_res) and len(np.unique(all_idx)) == len(y_train_res), "Overlap or missing samples"

clients_data = [Subset(balanced_train_ds, ci.tolist()) for ci in client_indices]
client_loaders = [DataLoader(cd, batch_size=32, shuffle=True,
                             generator=torch.Generator().manual_seed(SEED + k))
                  for k, cd in enumerate(clients_data)]
client_sample_sizes = [len(cd) for cd in clients_data]

client_table = pd.DataFrame(
    [np.bincount(y_train_res[ci], minlength=len(class_names)) for ci in client_indices],
    columns=class_names, index=[f"Client{k+1}" for k in range(NUM_CLIENTS)])
print(client_table)
print(f"\nSamples per client: min {min(client_sample_sizes)} | max {max(client_sample_sizes)}")

In [ ]:
# BLOCK 11 (UPDATED): IDS metrics, communication & privacy tracking
from typing import Iterable, Dict, List, Optional, Union
ArrayLike = Union[np.ndarray, torch.Tensor, List[float], List[int]]

def _to_np(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

# ---------- Multiclass + IDS metrics ----------
class MulticlassMetricsTracker:
    def __init__(self, num_classes: int, class_names: Optional[List[str]] = None, normal_class: str = 'normal'):
        self.num_classes = int(num_classes)
        self.class_names = class_names or [str(i) for i in range(self.num_classes)]
        assert len(self.class_names) == self.num_classes
        self.normal_idx = self.class_names.index(normal_class) if normal_class in self.class_names else None
        self.reset()

    def reset(self):
        self.cm = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)   # rows = true, cols = pred
        self.metrics_history: Dict[int, List[Dict[str, float]]] = {i: [] for i in range(self.num_classes)}

    def update(self, predictions: ArrayLike, labels: ArrayLike):
        p = _to_np(predictions).astype(np.int64).ravel()
        t = _to_np(labels).astype(np.int64).ravel()
        assert p.shape == t.shape
        np.add.at(self.cm, (t, p), 1)

    def update_from_logits(self, logits: torch.Tensor, labels: ArrayLike):
        self.update(torch.argmax(logits, dim=1), labels)

    @property
    def TP(self): return np.diag(self.cm)
    @property
    def FP(self): return self.cm.sum(0) - np.diag(self.cm)
    @property
    def FN(self): return self.cm.sum(1) - np.diag(self.cm)
    @property
    def TN(self): return self.cm.sum() - self.TP - self.FP - self.FN

    def get_metrics(self, store_history: bool = True) -> Dict[str, Dict[str, float]]:
        res = {}
        TP, FP, FN, TN = self.TP, self.FP, self.FN, self.TN
        for c in range(self.num_classes):
            far = FP[c] / (FP[c] + TN[c]) if (FP[c] + TN[c]) else 0.0
            dr  = TP[c] / (TP[c] + FN[c]) if (TP[c] + FN[c]) else 0.0
            acc = (TP[c] + TN[c]) / self.cm.sum() if self.cm.sum() else 0.0
            m = {'FAR': float(far), 'DR': float(dr), 'Accuracy': float(acc), 'Support': int(self.cm[c].sum())}
            if store_history: self.metrics_history[c].append(m)
            res[self.class_names[c]] = m
        return res

    def ids_metrics(self) -> Dict[str, float]:
        """IDS view: FAR = normal flagged as attack; DR = attacks flagged as any attack."""
        assert self.normal_idx is not None, "normal class not found"
        n = self.normal_idx
        normal_total   = self.cm[n].sum()
        normal_as_att  = normal_total - self.cm[n, n]
        attack_rows    = [i for i in range(self.num_classes) if i != n]
        attack_total   = self.cm[attack_rows].sum()
        attack_as_att  = attack_total - self.cm[attack_rows, n].sum()
        return {'IDS_FAR': float(normal_as_att / normal_total) if normal_total else 0.0,
                'IDS_DR':  float(attack_as_att / attack_total) if attack_total else 0.0,
                'Accuracy': float(np.trace(self.cm) / self.cm.sum()) if self.cm.sum() else 0.0}

    def overall_metrics(self) -> Dict[str, float]:
        m = self.ids_metrics()
        return {'FAR': m['IDS_FAR'], 'DR': m['IDS_DR'], 'Accuracy': m['Accuracy']}

# ---------- Communication tracking ----------
class CommunicationTracker:
    def __init__(self):
        self.total_bytes, self.rounds = 0, 0
        self.bytes_history: List[int] = []; self.clients_history: List[int] = []; self.kind_history: List[str] = []

    @staticmethod
    def _nbytes_of(p): return int(_to_np(p).nbytes)

    def track_round(self, parameters, n_clients: int, kind: str = "delta"):
        b = sum(self._nbytes_of(p) for p in parameters)
        self.total_bytes += b; self.rounds += 1
        self.bytes_history.append(b); self.clients_history.append(int(n_clients)); self.kind_history.append(kind)

    def track_communication(self, parameters):
        self.track_round(parameters, n_clients=0, kind="unknown")
        return self.bytes_history[-1]

    def get_efficiency(self) -> Dict[str, float]:
        avg = self.total_bytes / self.rounds if self.rounds else 0.0
        return {'avg_bytes_per_round': avg, 'avg_MB_per_round': avg / 2**20,
                'total_MB': self.total_bytes / 2**20, 'rounds': self.rounds,
                'avg_clients_per_round': float(np.mean(self.clients_history)) if self.clients_history else 0.0}

    def reset(self): self.__init__()

# ---------- Privacy & utility tracking ----------
class PrivacyTracker:
    def __init__(self, delta: float = 1e-5, ldp: bool = True):
        self.delta, self.ldp = float(delta), bool(ldp)
        self.reset()

    def reset(self):
        self.round_index: List[int] = []; self.epsilon_history: List[float] = []
        self.sigma_history: List[float] = []; self.clip_history: List[float] = []
        self.grad_norm_history: List[float] = []; self.accuracy_history: List[float] = []
        self.epsilon_total_history: List[float] = []; self.dp_enabled_history: List[bool] = []

    def track_round(self, epsilon_t, sigma_t, clip_norm, grad_norm, accuracy, *,
                    eps_total=None, round_idx=None, dp_enabled=None):
        self.epsilon_history.append(float(epsilon_t)); self.sigma_history.append(float(sigma_t))
        self.clip_history.append(float(clip_norm)); self.grad_norm_history.append(float(grad_norm))
        self.accuracy_history.append(float(accuracy))
        self.epsilon_total_history.append(float(eps_total) if eps_total is not None else float('nan'))
        self.dp_enabled_history.append(bool(dp_enabled) if dp_enabled is not None else False)
        self.round_index.append(int(round_idx) if round_idx is not None else len(self.round_index) + 1)

    def get_privacy_utility_state(self, round_idx=None) -> Dict[str, float]:
        acc = self.accuracy_history[-1] if self.accuracy_history else 50.0
        g = float(np.mean(self.grad_norm_history[-5:])) if self.grad_norm_history else 1.0
        return {"avg_grad_norm": g, "recent_accuracy": float(acc)}

    def get_privacy_utility_snapshot(self, k: int = 5) -> Dict[str, float]:
        return {'current_round': self.round_index[-1] if self.round_index else 0,
                'epsilon_t': self.epsilon_history[-1] if self.epsilon_history else 0.0,
                'epsilon_total': self.epsilon_total_history[-1] if self.epsilon_total_history else 0.0,
                'delta': self.delta,
                'recent_accuracy': float(np.mean(self.accuracy_history[-k:])) if self.accuracy_history else 0.0,
                'avg_grad_norm': float(np.mean(self.grad_norm_history[-k:])) if self.grad_norm_history else 1.0,
                'ldp': self.ldp,
                'dp_enabled': self.dp_enabled_history[-1] if self.dp_enabled_history else False}

# Quick self-test
_t = MulticlassMetricsTracker(len(class_names), class_names)
_t.update(y_test[:1000], y_test[:1000])
print("Self-test (perfect preds):", _t.ids_metrics())

In [ ]:
# BLOCK 12 (UPDATED): Real DDQN detector - trained ONLY with the DDQN (TD) loss
# MDP: state = flow features | action = predicted class | reward = +w[y] if correct, -w[y] if wrong
#      next state = next flow in the client's shuffled stream | episode = EPISODE_LEN flows
from typing import List, Iterable

class QNetwork(nn.Module):
    def __init__(self, input_dim: int, action_dim: int):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),       nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2),
        )
        self.q_head = nn.Linear(128, action_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)

    def forward(self, x, mode: str = "q_value"):
        # 'mode' kept only for compatibility with later blocks - ALWAYS returns Q-values.
        return self.q_head(self.feature_extractor(x))


class TensorReplayBuffer:
    def __init__(self, capacity: int, state_dim: int, device):
        self.cap, self.device, self.ptr, self.size = int(capacity), device, 0, 0
        self.s  = torch.zeros(self.cap, state_dim, device=device)
        self.s2 = torch.zeros(self.cap, state_dim, device=device)
        self.a  = torch.zeros(self.cap, dtype=torch.long, device=device)
        self.r  = torch.zeros(self.cap, device=device)
        self.d  = torch.zeros(self.cap, device=device)

    def push_batch(self, s, a, r, s2, d):
        n = s.shape[0]
        idx = (torch.arange(n, device=self.device) + self.ptr) % self.cap
        self.s[idx], self.a[idx], self.r[idx], self.s2[idx], self.d[idx] = s, a, r, s2, d
        self.ptr = (self.ptr + n) % self.cap
        self.size = min(self.size + n, self.cap)

    def sample(self, n: int):
        idx = torch.randint(0, self.size, (n,), device=self.device)
        return self.s[idx], self.a[idx], self.r[idx], self.s2[idx], self.d[idx]

    def __len__(self): return self.size


class DDQN:
    def __init__(self, input_dim, action_dim, device, class_weights,
                 gamma=0.1, lr=1e-3, eps_start=1.0, eps_end=0.05, eps_decay_steps=2000,
                 buffer_capacity=50_000, batch_size=256, tau=0.005, episode_len=32):
        self.device, self.action_dim = device, action_dim
        self.gamma, self.batch_size, self.tau, self.episode_len = gamma, batch_size, tau, episode_len
        self.eps_start, self.eps_end, self.eps_decay_steps = eps_start, eps_end, eps_decay_steps
        self.steps = 0

        self.online_net = QNetwork(input_dim, action_dim).to(device)
        self.target_net = QNetwork(input_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict()); self.target_net.eval()
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.loss_fn = nn.SmoothL1Loss()

        self.reward_w = torch.tensor(class_weights, dtype=torch.float32, device=device)
        self.memory = TensorReplayBuffer(buffer_capacity, input_dim, device)

    @property
    def epsilon(self):
        frac = min(1.0, self.steps / self.eps_decay_steps)
        return self.eps_start + frac * (self.eps_end - self.eps_start)

    @torch.no_grad()
    def act(self, states):
        self.online_net.eval()
        greedy = self.online_net(states).argmax(1)
        self.online_net.train()
        rand = torch.randint(0, self.action_dim, greedy.shape, device=self.device)
        explore = torch.rand(greedy.shape, device=self.device) < self.epsilon
        return torch.where(explore, rand, greedy)

    def reward(self, actions, labels):
        w = self.reward_w[labels]
        return torch.where(actions == labels, w, -w)

    @torch.no_grad()
    def collect(self, xb, yb):
        xb, yb = xb.to(self.device), yb.to(self.device)
        actions = self.act(xb)
        rewards = self.reward(actions, yb)
        next_states = torch.roll(xb, shifts=-1, dims=0)
        dones = ((torch.arange(len(xb), device=self.device) + 1) % self.episode_len == 0).float()
        dones[-1] = 1.0
        self.memory.push_batch(xb, actions, rewards, next_states, dones)
        return actions

    def td_loss(self):
        if len(self.memory) < self.batch_size:
            return None
        s, a, r, s2, d = self.memory.sample(self.batch_size)
        q = self.online_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            self.online_net.eval()
            best_next = self.online_net(s2).argmax(1, keepdim=True)          # online selects
            self.online_net.train()
            next_q = self.target_net(s2).gather(1, best_next).squeeze(1)    # target evaluates
            target = r + (1.0 - d) * self.gamma * next_q
        return self.loss_fn(q, target)

    def train_step(self, extra_loss=None, max_grad_norm=1.0):
        loss = self.td_loss()
        if loss is None:
            return None
        total = loss + (extra_loss() if extra_loss is not None else 0.0)
        self.optimizer.zero_grad(set_to_none=True)
        total.backward()
        torch.nn.utils.clip_grad_norm_(self.online_net.parameters(), max_grad_norm)
        self.optimizer.step()
        self.update_target()
        self.steps += 1
        return float(loss.item())

    @torch.no_grad()
    def update_target(self):
        for tp, p in zip(self.target_net.parameters(), self.online_net.parameters()):
            tp.data.lerp_(p.data, self.tau)

    @torch.no_grad()
    def predict(self, x):
        self.online_net.eval()
        out = self.online_net(x.to(self.device)).argmax(1)
        self.online_net.train()
        return out

    def get_fl_parameters(self) -> List[torch.Tensor]:
        return [p.detach().cpu().clone() for p in self.online_net.parameters()]

    @torch.no_grad()
    def set_fl_parameters(self, params: Iterable[torch.Tensor]):
        for p, new in zip(self.online_net.parameters(), params):
            p.data.copy_(new.to(p.device))
        self.target_net.load_state_dict(self.online_net.state_dict())

    @torch.no_grad()
    def param_vector(self):
        return torch.cat([p.detach().view(-1) for p in self.online_net.parameters()])


# ---------- Smoke test: does a single agent learn from reward alone? ----------
input_dim, action_dim = X_train.shape[1], len(class_names)
_agent = DDQN(input_dim, action_dim, device, class_weights, eps_decay_steps=500)
print("Input features:", input_dim, "| Parameters:", sum(p.numel() for p in _agent.online_net.parameters()))

for ep in range(10):
    losses = []
    for xb, yb in client_loaders[0]:
        _agent.collect(xb, yb)
        l = _agent.train_step()
        if l is not None: losses.append(l)
    print(f"epoch {ep+1:2d}: steps={_agent.steps:4d} eps={_agent.epsilon:.3f} "
          f"TD-loss={np.mean(losses) if losses else float('nan'):.4f}")

_m = MulticlassMetricsTracker(action_dim, class_names)
for xb, yb in DataLoader(val_dataset, batch_size=1024):
    _m.update(_agent.predict(xb), yb)
print("\nVal (single client, reward-only):", _m.ids_metrics())
for name, m in _m.get_metrics(store_history=False).items():
    print(f"{name:12s} DR={m['DR']:.3f}  support={m['Support']}")
del _agent, _m

In [ ]:
# BLOCK 13 (UPDATED): FL client - DDQN local training (reward only) + FedProx + client-level DP on Δ
import math

DP_ENABLED = False   # Run 1: non-private FL baseline. Privacy results may ONLY come from runs with True.

class FLClient:
    def __init__(self, ddqn, train_loader, device, model_name="Client",
                 default_mu=0.01, default_epsilon=1.0, default_clip_norm=1.0,
                 default_delta=1e-5, updates_per_batch=2):
        self.ddqn, self.train_loader = ddqn, train_loader
        self.device, self.model_name = torch.device(device), model_name
        self.mu, self.epsilon = float(default_mu), float(default_epsilon)
        self.clip_norm, self.delta = float(default_clip_norm), float(default_delta)
        self.updates_per_batch = int(updates_per_batch)
        self.global_params = None
        self.last_signal_norm = self.last_noise_norm = 0.0

    def set_parameters(self, parameters): self.ddqn.set_fl_parameters(parameters)
    def get_parameters(self): return self.ddqn.get_fl_parameters()

    # Standard FedProx: (mu/2) * SUM ||w - w_global||^2  (no division by #params)
    def _prox_term(self):
        snap = [g.to(self.device) for g in self.global_params]
        params = list(self.ddqn.online_net.parameters())
        def prox():
            return 0.5 * self.mu * sum(torch.sum((p - g) ** 2) for p, g in zip(params, snap))
        return prox

    def local_train(self, epochs):
        prox = self._prox_term() if (self.global_params is not None and self.mu > 0) else None
        losses = []
        for _ in range(max(1, int(epochs))):
            for xb, yb in self.train_loader:
                self.ddqn.collect(xb, yb)
                for _ in range(self.updates_per_batch):
                    l = self.ddqn.train_step(extra_loss=prox)
                    if l is not None: losses.append(l)
        return float(np.mean(losses)) if losses else float('nan')

    @torch.no_grad()
    def local_accuracy(self, max_samples=2048):
        correct = total = 0
        for xb, yb in self.train_loader:
            correct += int((self.ddqn.predict(xb).cpu() == yb).sum()); total += len(yb)
            if total >= max_samples: break
        return 100.0 * correct / max(1, total)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.global_params = [p.detach().cpu().clone() for p in self.ddqn.online_net.parameters()]
        self.epsilon   = float(config.get("epsilon",   self.epsilon))
        self.clip_norm = float(config.get("clip_norm", self.clip_norm))
        self.delta     = float(config.get("delta",     self.delta))
        self.mu        = float(config.get("mu",        self.mu))

        td_loss = self.local_train(int(config.get("local_epochs", 1)))

        with torch.no_grad():
            g_vec = torch.cat([g.view(-1) for g in self.global_params]).to(self.device)
            delta_vec = self.ddqn.param_vector() - g_vec
            delta_l2_raw = float(torch.norm(delta_vec))
            C = self.clip_norm
            sigma = C * math.sqrt(2 * math.log(1.25 / self.delta)) / max(self.epsilon, 1e-12)

            if DP_ENABLED:
                delta_clipped = delta_vec * min(1.0, C / (delta_l2_raw + 1e-12))
                noise = torch.normal(0.0, sigma, size=delta_clipped.shape, device=self.device)
                delta_private = delta_clipped + noise
                assert float(torch.norm(noise)) > 0, "DP guard: noise not applied"
            else:
                delta_clipped = delta_vec                       # non-private baseline: no clip, no noise
                noise = torch.zeros_like(delta_vec)
                delta_private = delta_vec

            self.last_signal_norm = float(torch.norm(delta_clipped))
            self.last_noise_norm  = float(torch.norm(noise))

            new_params, off = [], 0
            for g in self.global_params:
                n = g.numel()
                new_params.append((g.to(self.device) + delta_private[off:off + n].view_as(g)).cpu())
                off += n

        metrics = {
            "loss": td_loss, "accuracy": self.local_accuracy(),
            "grad_sensitivity": delta_l2_raw, "delta_l2": delta_l2_raw,
            "epsilon": self.epsilon, "sigma": sigma if DP_ENABLED else 0.0,
            "clip_norm": C, "mu": self.mu, "dp_enabled": DP_ENABLED,
            "signal_norm": self.last_signal_norm, "noise_norm": self.last_noise_norm,
        }
        return new_params, len(self.train_loader.dataset), metrics


# ---------- Smoke test: one local round on 2 clients ----------
_global = DDQN(input_dim, action_dim, device, class_weights)
_P = sum(p.numel() for p in _global.online_net.parameters())
for k in range(2):
    c = FLClient(DDQN(input_dim, action_dim, device, class_weights), client_loaders[k], device, model_name=f"Client{k+1}")
    _, n, m = c.fit(_global.get_fl_parameters(), {"local_epochs": 1, "mu": 0.01, "epsilon": 1.0, "clip_norm": 1.0})
    print(f"{c.model_name}: n={n} | TD-loss={m['loss']:.4f} | local acc={m['accuracy']:.1f}% | ‖Δ‖={m['delta_l2']:.3f} | DP={m['dp_enabled']}")

_sigma = 1.0 * math.sqrt(2 * math.log(1.25 / 1e-5)) / 1.0
print(f"\nIf DP were ON (eps=1, C=1): sigma={_sigma:.2f} -> expected noise norm ≈ {_sigma * math.sqrt(_P):.0f} vs signal ≤ 1.0")
del _global, c

# ExtraTree Feature Selection

In [ ]:
# BLOCK 14 (UPDATED): Server - aggregation, evaluation, one FL round (no client trimming)
import math

@torch.no_grad()
def evaluate_global(global_ddqn, dataset, batch_size=1024):
    m = MulticlassMetricsTracker(len(class_names), class_names)
    for xb, yb in DataLoader(dataset, batch_size=batch_size):
        m.update(global_ddqn.predict(xb), yb)
    return m

def weighted_average(client_params, client_sizes):
    total = float(sum(client_sizes))
    return [sum((n / total) * p[i] for p, n in zip(client_params, client_sizes))
            for i in range(len(client_params[0]))]

def federated_round(round_idx, global_ddqn, clients, rng, *, client_fraction, local_epochs,
                    mu, epsilon, clip_norm, delta, privacy_tracker=None, comm_tracker=None):
    k = max(1, math.ceil(client_fraction * len(clients)))
    active = rng.choice(len(clients), size=k, replace=False)
    global_params = global_ddqn.get_fl_parameters()

    params, sizes, mets = [], [], []
    for i in active:
        p, n, m = clients[i].fit(global_params, {"local_epochs": local_epochs, "mu": mu,
                                                 "epsilon": epsilon, "clip_norm": clip_norm, "delta": delta})
        params.append(p); sizes.append(n); mets.append(m)

    global_ddqn.set_fl_parameters(weighted_average(params, sizes))

    if comm_tracker is not None:
        comm_tracker.track_round([t for p in params for t in p], n_clients=k, kind="uplink")

    val = evaluate_global(global_ddqn, val_dataset).ids_metrics()
    med_delta = float(np.median([m["delta_l2"] for m in mets]))
    if privacy_tracker is not None:
        privacy_tracker.track_round(
            epsilon if DP_ENABLED else 0.0,
            float(np.mean([m["sigma"] for m in mets])),
            clip_norm, med_delta, 100.0 * val["Accuracy"],
            round_idx=round_idx, dp_enabled=DP_ENABLED)

    print(f"Round {round_idx:02d} | clients={k} | Acc={100*val['Accuracy']:.2f}% | "
          f"FAR={100*val['IDS_FAR']:.2f}% | DR={100*val['IDS_DR']:.2f}% | "
          f"med‖Δ‖={med_delta:.3f} | ‖noise‖={np.mean([m['noise_norm'] for m in mets]):.1f} | DP={DP_ENABLED}")
    return val, mets

In [ ]:
# BLOCK 15 (UPDATED): Final evaluation & plots - correct class names, Q-values as scores
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

@torch.no_grad()
def collect_predictions(global_ddqn, dataset, batch_size=1024):
    net = global_ddqn.online_net; net.eval()
    qs, ys = [], []
    for xb, yb in DataLoader(dataset, batch_size=batch_size):
        qs.append(net(xb.to(device)).cpu()); ys.append(yb)
    net.train()
    q = torch.cat(qs).numpy(); y_true = torch.cat(ys).numpy()
    return y_true, q.argmax(1), q

def final_report(global_ddqn, dataset, title="Test"):
    y_true, y_pred, q = collect_predictions(global_ddqn, dataset)
    m = MulticlassMetricsTracker(len(class_names), class_names); m.update(y_pred, y_true)
    ids = m.ids_metrics()
    print(f"\n===== {title} =====")
    print(f"Accuracy={100*ids['Accuracy']:.2f}% | IDS_FAR={100*ids['IDS_FAR']:.2f}% | IDS_DR={100*ids['IDS_DR']:.2f}%")
    print(classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                target_names=class_names, digits=3, zero_division=0))
    # One-vs-rest AUC using Q-values as ranking scores (Q-values are not probabilities)
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    aucs = {class_names[i]: float(roc_auc_score(y_bin[:, i], q[:, i]))
            for i in range(len(class_names)) if y_bin[:, i].any()}
    print("OvR AUC (Q-value scores):", {k: round(v, 4) for k, v in aucs.items()})
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion matrix - {title}')
    plt.tight_layout(); plt.show()
    return {"ids": ids, "auc": aucs, "cm": cm}

def plot_training_curves(history):
    r = [h['round'] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(r, [100 * h['Accuracy'] for h in history]); ax[0].set_title('Validation accuracy (%)'); ax[0].set_xlabel('Round')
    ax[1].plot(r, [100 * h['IDS_FAR'] for h in history], label='FAR')
    ax[1].plot(r, [100 * h['IDS_DR'] for h in history], label='DR')
    ax[1].set_title('IDS FAR / DR (%)'); ax[1].set_xlabel('Round'); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
# BLOCK 16 (v4 - C0): Record-level DP-SGD client + RDP privacy accounting
# - Each flow (record) belongs to exactly ONE client. Within a client it is Poisson-sampled many times
#   over the run (expected ROUNDS * LOCAL_STEPS * q times); the RDP accountant counts every step.
# - Per-sample gradients clipped to C, Gaussian noise N(0, (sigma*C)^2), Poisson sampling rate q = B / n
# - Privacy measured with the Opacus RDP accountant (no hand formula)
# - DP runs use one-step targets (gamma = 0): each loss term depends on one record only.
# C0 changes (see CHANGELOG.md):
#   - noise_multiplier = 0 -> no noise and NO accountant step; such a client reports epsilon = None (never a number)
#   - clip_norm = None -> no clipping; only allowed together with noise_multiplier = 0 (non-private baselines)
#   - projected_epsilon(sigma, steps) lives here so every runner uses the same pre-round budget check
import copy
import math
import torch.nn.functional as F
from torch.func import functional_call, vmap, grad
from opacus.accountants import RDPAccountant
from sklearn.utils.class_weight import compute_class_weight

# ---------- DP data: ALL training records, each in exactly one client (no under/oversampling) ----------
X_dp, y_dp = X_train.astype(np.float32), y_train.astype(np.int64)
cw_dp = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_dp)
class_weights_dp = (np.sqrt(cw_dp) / np.sqrt(cw_dp).mean()).astype(np.float32)

dp_client_indices = dirichlet_split(y_dp, NUM_CLIENTS, ALPHA, np.random.default_rng(SEED), MIN_PER_CLIENT)
_all = np.concatenate(dp_client_indices)
assert len(_all) == len(y_dp) and len(np.unique(_all)) == len(y_dp), "A record is in more than one client"
print("DP class weights:", dict(zip(class_names, np.round(class_weights_dp, 3))))
print("DP training data per class:", dict(zip(class_names, np.bincount(y_dp, minlength=len(class_names)))))
print("DP client sizes (seed", SEED, "):", [len(ci) for ci in dp_client_indices])


class DPClient:
    def __init__(self, ddqn, X_local, y_local, device, model_name="Client",
                 batch_size=64, clip_norm=1.0, delta=1e-5):
        self.ddqn, self.device, self.model_name = ddqn, device, model_name
        self.X = torch.as_tensor(X_local, dtype=torch.float32, device=device)
        self.y = torch.as_tensor(y_local, dtype=torch.long, device=device)
        self.n = len(self.y)
        self.batch_size, self.delta = batch_size, float(delta)
        self.C = None if clip_norm is None else float(clip_norm)
        self.q = min(1.0, batch_size / self.n)             # Poisson sampling rate
        self.accountant = RDPAccountant()
        self.nonprivate_steps = 0                          # steps taken with sigma = 0 (no privacy claim possible)
        self.mu, self.global_params = 0.0, None

    def epsilon_spent(self):
        if self.nonprivate_steps:
            return None                                    # any non-private step -> no epsilon may be reported
        return float(self.accountant.get_epsilon(delta=self.delta)) if self.accountant.history else 0.0

    def projected_epsilon(self, sigma, steps):
        """Epsilon this client WOULD have after `steps` more DP steps at `sigma` (nothing is spent)."""
        acc = copy.deepcopy(self.accountant)
        acc.history = list(acc.history) + [(float(sigma), self.q, int(steps))]
        return float(acc.get_epsilon(delta=self.delta))

    def _dp_step(self, sigma):
        sigma = float(sigma)
        assert sigma >= 0.0, "noise multiplier must be >= 0"
        if sigma > 0:
            assert self.C is not None, "DP guard: noise without clipping has unbounded sensitivity"
            self.accountant.step(noise_multiplier=sigma, sample_rate=self.q)   # counted even if batch is empty
        else:
            self.nonprivate_steps += 1

        net = self.ddqn.online_net
        idx = (torch.rand(self.n, device=self.device) < self.q).nonzero().squeeze(1)
        params  = {k: v.detach() for k, v in net.named_parameters()}
        buffers = {k: v.detach() for k, v in net.named_buffers()}

        if len(idx) > 0:
            x, y = self.X[idx], self.y[idx]
            a = self.ddqn.act(x)                          # epsilon-greedy action
            r = self.ddqn.reward(a, y)                    # one-step target (gamma = 0)
            net.train()

            def loss_fn(p, b, xi, ai, ri):
                qv = functional_call(net, (p, b), (xi.unsqueeze(0),)).squeeze(0)
                return F.smooth_l1_loss(qv.gather(0, ai.unsqueeze(0)).squeeze(0), ri)

            g = vmap(grad(loss_fn), in_dims=(None, None, 0, 0, 0), randomness="different")(params, buffers, x, a, r)
            B = len(idx)
            if self.C is not None:
                norms = torch.sqrt(sum(gi.reshape(B, -1).pow(2).sum(1) for gi in g.values()))
                factor = (self.C / (norms + 1e-12)).clamp(max=1.0)        # per-sample clipping
            else:
                factor = torch.ones(B, device=self.device)                 # non-private baseline: no clipping
            summed = {k: (gi * factor.view(-1, *([1] * (gi.dim() - 1)))).sum(0) for k, gi in g.items()}
        else:
            summed = {k: torch.zeros_like(v) for k, v in params.items()}

        expected_bs = self.q * self.n
        for k, p in net.named_parameters():
            p.grad = summed[k] / expected_bs
            if sigma > 0:
                p.grad = p.grad + torch.normal(0.0, sigma * self.C, size=p.shape, device=self.device) / expected_bs
            if self.global_params is not None and self.mu > 0:          # FedProx: data-independent, no privacy cost
                p.grad = p.grad + self.mu * (p.detach() - self.global_params[k])
        self.ddqn.optimizer.step()
        self.ddqn.optimizer.zero_grad(set_to_none=True)
        self.ddqn.update_target()
        self.ddqn.steps += 1

    @torch.no_grad()
    def local_accuracy(self, max_samples=2048):
        xs, ys = self.X[:max_samples], self.y[:max_samples]
        return 100.0 * float((self.ddqn.predict(xs) == ys).float().mean())

    def fit(self, parameters, config):
        self.ddqn.set_fl_parameters(parameters)
        net = self.ddqn.online_net
        self.global_params = {k: v.detach().clone() for k, v in net.named_parameters()}
        self.mu = float(config.get("mu", 0.0))
        sigma = float(config["noise_multiplier"])
        for _ in range(int(config["local_steps"])):
            self._dp_step(sigma)
        # Aggregation of a DP-trained model is post-processing: no extra noise needed here
        metrics = {"accuracy": self.local_accuracy(), "sigma": sigma,
                   "epsilon_spent": self.epsilon_spent(), "dp_enabled": sigma > 0,
                   "delta_l2": float(torch.norm(self.ddqn.param_vector() -
                                     torch.cat([v.view(-1) for v in self.global_params.values()])))}
        return self.ddqn.get_fl_parameters(), self.n, metrics


In [ ]:
# BLOCK 16b (v4 - speed): all clients' local DP-SGD steps in parallel on the GPU (same algorithm, far fewer GPU calls)
# For EACH client k, one step is exactly DPClient._dp_step:
#   Poisson sampling with the client's own rate q_k -> epsilon-greedy action -> one-step reward target ->
#   per-sample gradient clipped to C -> + N(0, (sigma_k * C)^2) noise -> / expected batch q_k * n_k ->
#   + FedProx term mu * (w - w_global) -> Adam.
# Adam is element-wise, so ONE Adam over the stacked [K, ...] tensors is K independent per-client Adams.
# Privacy accounting stays in each client's own RDP accountant (DPClient), step by step as before.
# Needs every client to train every round (CLIENT_FRAC = 1.0, the thesis setting); otherwise the per-client loop runs.
from torch.func import stack_module_state


class FastFederation:
    def __init__(self, clients):
        self.clients, self.K = clients, len(clients)
        c0 = clients[0]
        self.dev, self.C = c0.device, c0.C
        assert all(c.C == self.C for c in clients), "all clients must share the clip norm"
        nets = [c.ddqn.online_net for c in clients]
        self.names = [n for n, _ in nets[0].named_parameters()]
        params, buffers = stack_module_state(nets)
        self.params = {k: v.detach().clone().requires_grad_(True) for k, v in params.items()}
        self.buffers = {k: v.detach().clone() for k, v in buffers.items()}
        self.base = copy.deepcopy(nets[0]).to("meta")
        self.opt = optim.Adam([self.params[n] for n in self.names], lr=c0.ddqn.optimizer.param_groups[0]["lr"])
        self.ag = c0.ddqn                                   # epsilon schedule + reward (same weights for all clients)
        self.steps = c0.ddqn.steps
        self.n = torch.tensor([c.n for c in clients], device=self.dev)
        self.q = torch.tensor([c.q for c in clients], device=self.dev, dtype=torch.float32)
        nmax = int(self.n.max())
        self.X = torch.zeros(self.K, nmax, c0.X.shape[1], device=self.dev)
        self.Y = torch.zeros(self.K, nmax, dtype=torch.long, device=self.dev)
        for k, c in enumerate(clients):
            self.X[k, :c.n], self.Y[k, :c.n] = c.X, c.y
        self.row_ok = torch.arange(nmax, device=self.dev)[None, :] < self.n[:, None]

        base = self.base
        def q_all(p, b, x):                                 # Q-values of one client's batch
            return functional_call(base, (p, b), (x,))
        def loss_one(p, b, xi, ai, ri):                     # one record's loss (one-step target, gamma = 0)
            qv = functional_call(base, (p, b), (xi.unsqueeze(0),)).squeeze(0)
            return F.smooth_l1_loss(qv.gather(0, ai.unsqueeze(0)).squeeze(0), ri)
        self._q = vmap(q_all)
        self._g = vmap(vmap(grad(loss_one), in_dims=(None, None, 0, 0, 0)), in_dims=(0, 0, 0, 0, 0))
        self.last_grad = None

    def _epsilon(self):
        a = self.ag
        return a.eps_start + min(1.0, self.steps / a.eps_decay_steps) * (a.eps_end - a.eps_start)

    def _step(self, sigma, mu, gp):
        K, dev = self.K, self.dev
        take = (torch.rand(self.row_ok.shape, device=dev) < self.q[:, None]) & self.row_ok   # Poisson, per client
        cnt = take.sum(1)
        B = int(cnt.max())
        if B > 0:
            order = torch.argsort((~take).to(torch.int8), dim=1, stable=True)[:, :B]
            valid = torch.arange(B, device=dev)[None, :] < cnt[:, None]
            x = torch.gather(self.X, 1, order[..., None].expand(-1, -1, self.X.shape[2]))
            y = torch.gather(self.Y, 1, order)
            p_det = {k: v.detach() for k, v in self.params.items()}
            with torch.no_grad():
                greedy = self._q(p_det, self.buffers, x).argmax(-1)
                explore = torch.rand(greedy.shape, device=dev) < self._epsilon()
                a = torch.where(explore, torch.randint(0, action_dim, greedy.shape, device=dev), greedy)
                r = self.ag.reward(a, y)
            g = self._g(p_det, self.buffers, x, a, r)                                       # [K, B, ...]
            if self.C is not None:
                norms = torch.sqrt(sum(gi.reshape(K, B, -1).pow(2).sum(-1) for gi in g.values()))
                factor = (self.C / (norms + 1e-12)).clamp(max=1.0)
            else:
                factor = torch.ones(K, B, device=dev)
            factor = factor * valid
            summed = {k: (gi * factor.view(K, B, *([1] * (gi.dim() - 2)))).sum(1) for k, gi in g.items()}
        else:
            summed = {k: torch.zeros_like(v) for k, v in self.params.items()}
        ebs = self.q * self.n
        grads = {}
        for k in self.names:
            p = self.params[k]
            shape = (K,) + (1,) * (p.dim() - 1)
            gk = summed[k]
            if bool((sigma > 0).any()):
                gk = gk + torch.randn_like(p) * (sigma * self.C).view(shape)
            gk = gk / ebs.view(shape)
            if mu > 0:
                gk = gk + mu * (p.detach() - gp[k])
            p.grad = gk
            grads[k] = gk
        self.last_grad = grads
        self.opt.step()
        self.opt.zero_grad(set_to_none=True)
        self.steps += 1

    def fit(self, gp_list, sigmas, steps, mu):
        """gp_list: global parameters (list, in parameter order). sigmas: one noise multiplier per client.
        Returns (params per client, sizes, metrics per client) like DPClient.fit."""
        sig = torch.tensor([float(s) for s in sigmas], device=self.dev)
        assert bool((sig >= 0).all()), "noise multiplier must be >= 0"
        if bool((sig > 0).any()):
            assert self.C is not None, "DP guard: noise without clipping has unbounded sensitivity"
        gp = {k: t.to(self.dev) for k, t in zip(self.names, gp_list)}
        with torch.no_grad():
            for k in self.names:
                self.params[k].copy_(gp[k].unsqueeze(0).expand_as(self.params[k]))
        for c, s in zip(self.clients, sigmas):           # accounting: identical to DPClient._dp_step, every step
            for _ in range(int(steps)):
                if s > 0:
                    c.accountant.step(noise_multiplier=float(s), sample_rate=c.q)
                else:
                    c.nonprivate_steps += 1
        for _ in range(int(steps)):
            self._step(sig, mu, gp)
        out, mets = [], []
        with torch.no_grad():
            gvec = torch.cat([gp[k].reshape(-1) for k in self.names])
            for i in range(self.K):
                p_i = [self.params[k][i].detach().cpu().clone() for k in self.names]
                out.append(p_i)
                d = float(torch.norm(torch.cat([t.reshape(-1) for t in p_i]).to(self.dev) - gvec))
                mets.append({"sigma": float(sigmas[i]), "epsilon_spent": self.clients[i].epsilon_spent(),
                             "dp_enabled": float(sigmas[i]) > 0, "delta_l2": d})
        return out, [c.n for c in self.clients], mets


def fit_clients(clients, active, gp, sigmas, steps, mu, fast=None):
    """Local training of the active clients. sigmas: dict client index -> noise multiplier.
    Uses FastFederation when every client is active; otherwise the per-client loop (identical algorithm)."""
    active = [int(i) for i in active]
    if fast is not None and sorted(active) == list(range(len(clients))):
        return fast.fit(gp, [sigmas[i] for i in range(len(clients))], steps, mu)
    params, sizes, mets = [], [], []
    for i in active:
        p, n, m = clients[i].fit(gp, {"noise_multiplier": sigmas[i], "local_steps": steps, "mu": mu})
        params.append(p); sizes.append(n); mets.append(m)
    return params, sizes, mets


In [ ]:
# BLOCK 17 (NEW - Chapter 6): ADP-Meta controller - REINFORCE policy over noise levels
# state  s_t = [t/T, Acc_{t-1}, dAcc_{t-1}, eps_spent / eps_budget]
# action a_t = noise multiplier from LEVEL_MULTS (larger = more noise = more privacy)
# reward r_t = dAcc_t - LAMBDA * (eps_t - eps_{t-1}) / eps_budget
# update: REINFORCE with moving-average baseline + entropy bonus
class ADPMetaController(nn.Module):
    def __init__(self, sigma_levels, state_dim=4, lr=1e-2, baseline_beta=0.9, entropy_coef=0.01, device="cpu"):
        super().__init__()
        self.sigma_levels = list(sigma_levels)
        self.device = torch.device(device)
        self.policy = nn.Sequential(nn.Linear(state_dim, 32), nn.Tanh(),
                                    nn.Linear(32, len(self.sigma_levels))).to(self.device)
        self.opt = optim.Adam(self.policy.parameters(), lr=lr)
        self.baseline, self.beta, self.entropy_coef = 0.0, baseline_beta, entropy_coef
        self._log_prob = self._entropy = None
        self.history = []                                   # (round, state, action, level, reward)

    def select(self, state):
        s = torch.as_tensor(state, dtype=torch.float32, device=self.device)
        dist = torch.distributions.Categorical(logits=self.policy(s))
        a = dist.sample()
        self._log_prob, self._entropy = dist.log_prob(a), dist.entropy()
        self._last = (list(map(float, state)), int(a))
        return int(a), self.sigma_levels[int(a)]

    def update(self, reward, round_idx):
        adv = float(reward) - self.baseline
        loss = -adv * self._log_prob - self.entropy_coef * self._entropy
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.baseline = self.beta * self.baseline + (1 - self.beta) * float(reward)
        s, a = self._last
        self.history.append({"round": round_idx, "state": s, "action": a,
                             "sigma": self.sigma_levels[a], "reward": float(reward)})

In [ ]:
# BLOCK 17b (v4 - C5/C6): ADP-Meta v2 - budget pacing + macro-F1 reward + meta-training on validation
# C5 (CFG["ADP_ACTIONS"] = "pacing"):
#   - Budget pacing in RDP at a fixed order alpha* per client. RDP at a fixed order ADDS UP over rounds, so
#     "spend a share of what is left" is exact. alpha* and the budget R come from the constant-noise plan
#     that spends exactly eps_budget over all rounds -> final Opacus epsilon <= eps_budget.
#   - Action k in PACE_ACTIONS: this round spends min(1, k / rounds_left) of the remaining RDP budget.
#     k = 1 every round reproduces fixed noise exactly; the last round always spends what is left.
#     sigma for the round is found by bisection, always rounding towards MORE noise.
#   - Policy network 32-16 (as in Chapter 6). State = [t/T, budget left, val macro-F1, d macro-F1,
#     log(1 + median client update norm)]. Reward r_t = validation macro-F1 gain of round t.
# C6 (CFG["ADP_META_EPISODES"] > 0):
#   - Practice episodes on SERVER-HELD VALIDATION data only (threat model): validation is split once into two
#     stratified halves (seed 42). Proxy detectors train on half A with the same DP-SGD client, batch size,
#     clip norm, local steps, rounds, FedProx mu and reward weights as the real run. The controller's reward
#     is measured on half B.
#   - "Same noise levels": proxy client k uses the noise schedule of REAL client k (its sampling rate q and
#     budget), so per-step noise sigma*C/B is identical to the real run for the same actions. The proxy
#     clients' own epsilon is meaningless (validation is public) and is never reported.
#   - REINFORCE with Monte-Carlo returns and a per-round baseline; one update per episode.
#   - The trained controller is frozen and used greedily in the real run; one file per eps budget,
#     shared by every seed (it is a fixed artifact, like a hyperparameter).
#   - The reported detector never trains on validation data; the test set is never used.
from opacus.accountants.analysis.rdp import compute_rdp, get_privacy_spent
from sklearn.model_selection import train_test_split

RDP_ORDERS = RDPAccountant.DEFAULT_ALPHAS


class RDPPacer:
    """Budget pacing for ONE client: RDP spent at a fixed order alpha* is additive across rounds."""
    def __init__(self, q, eps_budget, delta, rounds, steps):
        self.q, self.steps = float(q), int(steps)
        self.sigma0 = get_noise_multiplier(target_epsilon=eps_budget, target_delta=delta, sample_rate=self.q,
                                           steps=rounds * steps, accountant="rdp")
        rdp = compute_rdp(q=self.q, noise_multiplier=self.sigma0, steps=rounds * steps, orders=RDP_ORDERS)
        _, self.alpha = get_privacy_spent(orders=RDP_ORDERS, rdp=rdp, delta=delta)
        self.R = self.round_rdp(self.sigma0) * rounds              # RDP budget at alpha*
        self.spent = 0.0

    def round_rdp(self, sigma):
        return float(compute_rdp(q=self.q, noise_multiplier=sigma, steps=self.steps, orders=[self.alpha])[0])

    @property
    def left_frac(self):
        return max(0.0, 1.0 - self.spent / self.R)

    def sigma_for_share(self, share):
        """Smallest sigma (up to bisection precision, rounded towards more noise) whose round cost <= share * left."""
        target = share * (self.R - self.spent)
        if target <= 1e-12 * self.R:
            return None
        lo, hi = 1e-2, max(1.0, self.sigma0)
        while self.round_rdp(hi) > target:
            hi *= 2.0
        for _ in range(60):
            mid = math.sqrt(lo * hi)
            if self.round_rdp(mid) > target:
                lo = mid
            else:
                hi = mid
        return hi

    def commit(self, sigma):
        self.spent += self.round_rdp(sigma)


def pacing_share(k, rounds_left):
    return 1.0 if rounds_left <= 1 else min(1.0, k / rounds_left)


class ADPMetaV2(nn.Module):
    STATE_DIM = 5

    def __init__(self, actions, lr=1e-2, entropy_coef=0.01, baseline_beta=0.8, device="cpu"):
        super().__init__()
        self.actions, self.device = list(actions), torch.device(device)
        self.policy = nn.Sequential(nn.Linear(self.STATE_DIM, 32), nn.Tanh(),
                                    nn.Linear(32, 16), nn.Tanh(),
                                    nn.Linear(16, len(self.actions))).to(self.device)
        self.opt = optim.Adam(self.policy.parameters(), lr=lr)
        self.entropy_coef, self.beta = entropy_coef, baseline_beta
        self.baseline = {}                   # per-round baseline of the return (episodic mode)
        self.online_baseline = 0.0           # moving baseline of the reward (online mode)
        self._logp, self._ent = [], []
        self.online_mode = False             # set by the runner: True -> update_online after every round

    @staticmethod
    def make_state(t, T, left_frac, f1, df1, upd_norm):
        return [t / T, left_frac, f1, df1, math.log1p(max(0.0, upd_norm))]

    def act(self, state, greedy=False):
        logits = self.policy(torch.as_tensor(state, dtype=torch.float32, device=self.device))
        if greedy:
            with torch.no_grad():
                return int(logits.argmax())
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        self._logp.append(dist.log_prob(a)); self._ent.append(dist.entropy())
        return int(a)

    def finish_episode(self, rewards):
        """REINFORCE with Monte-Carlo returns G_t = sum_{s>=t} r_s and a per-round EMA baseline."""
        assert len(rewards) == len(self._logp), "one reward per sampled action"
        G = np.cumsum(np.asarray(rewards, dtype=np.float64)[::-1])[::-1]
        adv = [g - self.baseline.get(t, g) for t, g in enumerate(G)]       # first episode: advantage 0
        loss = -sum(a * lp for a, lp in zip(adv, self._logp)) - self.entropy_coef * sum(self._ent)
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        for t, g in enumerate(G):
            self.baseline[t] = self.beta * self.baseline.get(t, g) + (1 - self.beta) * g
        self._logp, self._ent = [], []
        return float(G[0]) if len(G) else 0.0

    def update_online(self, reward):
        """Online variant (C5 without meta-training): one REINFORCE step per round."""
        adv = float(reward) - self.online_baseline
        loss = -adv * self._logp[-1] - self.entropy_coef * self._ent[-1]
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.online_baseline = self.beta * self.online_baseline + (1 - self.beta) * float(reward)
        self._logp, self._ent = [], []


def _drop_unrewarded(ctrl, rewards):
    """The round that is not run gets no reward -> forget the action just sampled for it (none if greedy)."""
    if len(ctrl._logp) > (len(rewards) if not ctrl.online_mode else 0):
        ctrl._logp.pop(); ctrl._ent.pop()


def paced_episode(name, g, clients, pacers, ctrl, eval_ds, *, rounds, steps, mu, rng, mode,
                  eps_budget=None, keep_states=False, fast=None):
    """One federated run whose noise each round is set by ADP-Meta v2 pacing.
    mode: "sample_episodic" (practice), "sample_online" (C5 real run), "greedy" (frozen, C6 real run).
    eps_budget: if given (real run), every client's Opacus epsilon is checked BEFORE spending and after."""
    assert mode in ("sample_episodic", "sample_online", "greedy")
    ctrl.online_mode = mode == "sample_online"
    history, states, rewards = [], [], []
    m = eval_metrics(g, eval_ds)
    f1_prev, df1_prev, upd = m["Macro_F1"], 0.0, 0.0
    for r in range(1, rounds + 1):
        left = float(np.mean([p.left_frac for p in pacers]))
        a = ctrl.act(ADPMetaV2.make_state(r, rounds, left, f1_prev, df1_prev, upd), greedy=(mode == "greedy"))
        k = ctrl.actions[a]
        share = pacing_share(k, rounds - r + 1)
        active = rng.choice(len(clients), size=max(1, math.ceil(CFG["CLIENT_FRAC"] * len(clients))), replace=False)
        sig = {int(i): pacers[i].sigma_for_share(share) for i in active}
        if any(s is None for s in sig.values()):
            print(f"[{name}] Round {r}: budget exhausted - stopping."); _drop_unrewarded(ctrl, rewards); break
        if eps_budget is not None and max(clients[i].projected_epsilon(sig[i], steps) for i in sig) > eps_budget + 1e-9:
            print(f"[{name}] Round {r}: privacy budget would be exceeded - stopping before spending.")
            _drop_unrewarded(ctrl, rewards); break
        gp = g.get_fl_parameters()
        params, sizes, mets = fit_clients(clients, list(sig), gp, sig, steps, mu, fast)
        for i, s in sig.items():
            pacers[i].commit(s)
        norms = [m["delta_l2"] for m in mets]
        g.set_fl_parameters(weighted_average(params, sizes))
        h = eval_metrics(g, eval_ds)
        reward = h["Macro_F1"] - f1_prev
        if mode == "sample_online":
            ctrl.update_online(reward)
        rewards.append(reward)
        eps_now = None
        if eps_budget is not None:
            eps_now = max(c.epsilon_spent() for c in clients)
            assert eps_now <= eps_budget + 1e-9, "DP guard: budget exceeded"
        h.update({"round": r, "action_k": k, "share": share, "median_sigma": float(np.median(list(sig.values()))),
                  "budget_left": float(np.mean([p.left_frac for p in pacers])), "reward": reward,
                  "epsilon": eps_now, "median_update_norm": float(np.median(norms))})
        history.append(h)
        if keep_states:
            states.append([t.clone() for t in g.get_fl_parameters()])
        _log_round(name, r, h, f" | k={k} share={share:.3f} sigma~{h['median_sigma']:.2f}"
                               + ("" if eps_now is None else f" | eps={eps_now:.3f}"))
        f1_prev, df1_prev, upd = h["Macro_F1"], reward, h["median_update_norm"]
    return history, states, rewards


def val_halves():
    """Fixed stratified split of the server-held validation set (seed 42): A = proxy training, B = reward."""
    ia, ib = train_test_split(np.arange(len(y_val)), test_size=0.5, stratify=y_val, random_state=SPLIT_SEED)
    return ia, ib


def controller_path(eps_budget):
    tag = f"__{CFG['TAG']}" if CFG.get("TAG") else ""
    return os.path.join(CFG["RESULTS_DIR"], f"controller_eps{eps_budget:g}{tag}.pt")


def meta_train_controller(eps_budget, C):
    """C6: train ADP-Meta v2 over practice episodes on validation half A (reward on half B). Cached per eps."""
    path = controller_path(eps_budget)
    ctrl = ADPMetaV2(CFG["PACE_ACTIONS"], lr=CFG["ADP_LR"], entropy_coef=CFG["ADP_ENTROPY"], device=device)
    if os.path.exists(path):
        ctrl.load_state_dict(torch.load(path)); print(f"Loaded meta-trained controller {path}")
        return ctrl
    ia, ib = val_halves()
    XA, yA = X_val[ia], y_val[ia]
    val_B = TensorDataset(torch.from_numpy(X_val[ib]), torch.from_numpy(y_val[ib]).long())
    q_real = [min(1.0, CFG["BATCH"] / len(ci)) for ci in dp_client_indices]      # real clients, seed-42 split
    R, STEPS, mu = CFG["ROUNDS"], CFG["LOCAL_STEPS"], CFG["MU"]
    set_seed(SPLIT_SEED)
    ctrl = ADPMetaV2(CFG["PACE_ACTIONS"], lr=CFG["ADP_LR"], entropy_coef=CFG["ADP_ENTROPY"], device=device)
    log, t0 = [], time.time()
    for e in range(CFG["ADP_META_EPISODES"]):
        es = 10_000 + e
        set_seed(es)
        parts = dirichlet_split(yA, NUM_CLIENTS, ALPHA, np.random.default_rng(es), CFG["PROXY_MIN_PER_CLIENT"])
        mk = lambda: SmallDDQN(input_dim, action_dim, device, reward_weights(), gamma=0.0, lr=CFG["LR"])
        g = mk()
        proxies = [DPClient(mk(), XA[ci], yA[ci], device, model_name=f"Proxy{k+1}", batch_size=CFG["BATCH"],
                            clip_norm=C, delta=CFG["DELTA"]) for k, ci in enumerate(parts)]
        pacers = [RDPPacer(q_real[k], eps_budget, CFG["DELTA"], R, STEPS) for k in range(len(proxies))]
        hist, _, rewards = paced_episode(f"meta eps{eps_budget:g} ep{e+1}", g, proxies, pacers, ctrl, val_B,
                                         rounds=R, steps=STEPS, mu=mu, rng=np.random.default_rng(es),
                                         mode="sample_episodic", fast=make_fast(proxies))
        ret = ctrl.finish_episode(rewards) if rewards else 0.0
        best = max((h["Macro_F1"] for h in hist), default=float("nan"))
        log.append({"episode": e + 1, "return": ret, "best_val_B_F1": best, "actions": [h["action_k"] for h in hist]})
        print(f"=== meta-episode {e+1}/{CFG['ADP_META_EPISODES']} | return={ret:+.4f} | best F1(B)={100*best:.1f} "
              f"| actions={log[-1]['actions']} | {(time.time()-t0)/60:.0f} min")
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    torch.save(ctrl.state_dict(), path)
    with open(path.replace(".pt", "_meta_log.json"), "w") as f:
        json.dump({"eps_budget": eps_budget, "episodes": log, "minutes": (time.time() - t0) / 60,
                   "val_halves_seed": SPLIT_SEED, "cfg": {k: (list(v) if isinstance(v, tuple) else v)
                                                          for k, v in CFG.items()}}, f, indent=2)
    return ctrl


In [ ]:
# BLOCK 17c (v4 - ADP-Meta v3): joint noise pacing + adaptive clipping, twin-baseline meta-training
# Same DDQN, FedProx, record-level local DP-SGD and RDP accounting as before. What changes is the controller:
#  (1) Each round it picks BOTH a budget-pacing action k (share of the remaining RDP budget, Block 17b) AND a clip
#      multiplier m: C_t = C_base * m. Privacy depends only on the noise multiplier sigma (noise std = sigma * C_t),
#      so changing C costs no privacy. The decision uses only validation metrics and already-noised updates.
#  (2) Meta-training is paired: each practice episode has a TWIN that uses the fixed schedule (k = 1, m = 1) with
#      the same proxy data, initialisation and sampling seed. The controller is rewarded for beating its twin.
#      Twins are cached per (seed, eps), so 6 twin runs serve all practice episodes.
#  (3) Reward = the episode's selected-round score under the SAME rule used for model selection
#      (best val macro-F1 with FAR <= cap; if the cap is never met, F1 minus the FAR excess).
#  (4) Richer state: t/T, budget left, macro-F1, d macro-F1, FAR, log(1 + update norm), previous log sigma,
#      previous log clip multiplier, log10(eps budget).
#  (5) More practice episodes (cheap with Block 16b). ONE controller for all eps budgets (eps is in the state).
# The policy is 32-16 with two heads. Its initial bias favours (k = 1, m = 1), i.e. it starts at the fixed
# schedule and explores around it. The real run uses the frozen policy greedily.
# Practice data = validation half A (proxy clients follow the real clients' noise schedule), reward on half B.
# The reported detector never trains on validation data, and the test set is never used.


class ADPMetaV3(nn.Module):
    STATE_DIM = 9

    def __init__(self, pace_actions, clip_actions, lr=1e-2, entropy_coef=0.01, baseline_beta=0.8, init_bias=1.0):
        super().__init__()
        self.pace, self.clip = list(pace_actions), list(clip_actions)
        self.trunk = nn.Sequential(nn.Linear(self.STATE_DIM, 32), nn.Tanh(), nn.Linear(32, 16), nn.Tanh())
        self.head_k, self.head_m = nn.Linear(16, len(self.pace)), nn.Linear(16, len(self.clip))
        with torch.no_grad():                                   # start at the fixed schedule (k = 1, m = 1)
            if 1.0 in self.pace:
                self.head_k.bias[self.pace.index(1.0)] += init_bias
            if 1.0 in self.clip:
                self.head_m.bias[self.clip.index(1.0)] += init_bias
        self.opt = optim.Adam(self.parameters(), lr=lr)
        self.entropy_coef, self.beta, self.baseline = entropy_coef, baseline_beta, None   # set by 1st episode
        self.gen = torch.Generator().manual_seed(SPLIT_SEED)    # own RNG: does not shift the training RNG streams
        self._logp, self._ent = [], []

    @staticmethod
    def make_state(t, T, left, f1, df1, far, upd, prev_sigma, prev_m, eps_budget):
        return [t / T, left, f1, df1, far, math.log1p(max(0.0, upd)), math.log(max(prev_sigma, 1e-3)),
                math.log(prev_m), math.log10(eps_budget)]

    def _dists(self, state):
        z = self.trunk(torch.as_tensor(state, dtype=torch.float32))
        return torch.softmax(self.head_k(z), -1), torch.softmax(self.head_m(z), -1)

    def act(self, state, greedy=False):
        pk, pm = self._dists(state)
        if greedy:
            return self.pace[int(pk.argmax())], self.clip[int(pm.argmax())]
        ik = int(torch.multinomial(pk.detach(), 1, generator=self.gen))
        im = int(torch.multinomial(pm.detach(), 1, generator=self.gen))
        self._logp.append(torch.log(pk[ik]) + torch.log(pm[im]))
        self._ent.append(-(pk * torch.log(pk + 1e-12)).sum() - (pm * torch.log(pm + 1e-12)).sum())
        return self.pace[ik], self.clip[im]

    def finish_episode(self, advantage):
        """Terminal reward: every decision of the episode shares the advantage over the twin."""
        if not self._logp:
            return
        if self.baseline is None:
            self.baseline = float(advantage)                    # first episode: no update signal, sets the baseline
        a = float(advantage) - self.baseline
        loss = -a * torch.stack(self._logp).sum() - self.entropy_coef * torch.stack(self._ent).sum()
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.baseline = self.beta * self.baseline + (1 - self.beta) * float(advantage)
        self._logp, self._ent = [], []


def episode_score(history):
    """Score of an episode under the model-selection rule (same rule as the reported runs)."""
    if not history:
        return -1.0
    i, ok = select_best_round(history, CFG["FAR_CAP"])
    h = history[i]
    return h["Macro_F1"] - (0.0 if ok else (h["FAR"] - CFG["FAR_CAP"]))


def paced_clip_episode(name, g, clients, pacers, policy, eval_ds, *, rounds, steps, mu, rng, C_base, eps_budget,
                       fast=None, check_budget=False, keep_states=False):
    """One federated run; each round policy(state) -> (k, m) sets the noise share and the clip norm C_base * m."""
    history, states = [], []
    m0 = eval_metrics(g, eval_ds)
    f1_prev, df1_prev, far_prev, upd, sig_prev, m_prev = m0["Macro_F1"], 0.0, m0["FAR"], 0.0, 1.0, 1.0
    for r in range(1, rounds + 1):
        left = float(np.mean([p.left_frac for p in pacers]))
        k, m = policy(ADPMetaV3.make_state(r, rounds, left, f1_prev, df1_prev, far_prev, upd, sig_prev, m_prev,
                                           eps_budget))
        share = pacing_share(k, rounds - r + 1)
        active = rng.choice(len(clients), size=max(1, math.ceil(CFG["CLIENT_FRAC"] * len(clients))), replace=False)
        sig = {int(i): pacers[i].sigma_for_share(share) for i in active}
        if any(s is None for s in sig.values()):
            print(f"[{name}] Round {r}: budget exhausted - stopping."); break
        if check_budget and max(clients[i].projected_epsilon(sig[i], steps) for i in sig) > eps_budget + 1e-9:
            print(f"[{name}] Round {r}: privacy budget would be exceeded - stopping before spending."); break
        C_t = C_base * m
        for c in clients:
            c.C = C_t                                           # clip norm for this round (sigma * C_t noise)
        if fast is not None:
            fast.C = C_t
        gp = g.get_fl_parameters()
        params, sizes, mets = fit_clients(clients, list(sig), gp, sig, steps, mu, fast)
        for i, s in sig.items():
            pacers[i].commit(s)
        g.set_fl_parameters(weighted_average(params, sizes))
        h = eval_metrics(g, eval_ds)
        eps_now = None
        if check_budget:
            eps_now = max(c.epsilon_spent() for c in clients)
            assert eps_now <= eps_budget + 1e-9, "DP guard: budget exceeded"
        h.update({"round": r, "action_k": k, "clip_mult": m, "clip": C_t, "share": share,
                  "median_sigma": float(np.median(list(sig.values()))), "epsilon": eps_now,
                  "budget_left": float(np.mean([p.left_frac for p in pacers])),
                  "median_update_norm": float(np.median([x["delta_l2"] for x in mets]))})
        history.append(h)
        if keep_states:
            states.append([t.clone() for t in g.get_fl_parameters()])
        _log_round(name, r, h, f" | k={k} m={m} C={C_t:.2f} sigma~{h['median_sigma']:.2f}"
                               + ("" if eps_now is None else f" | eps={eps_now:.3f}"))
        f1_prev, df1_prev, far_prev = h["Macro_F1"], h["Macro_F1"] - f1_prev, h["FAR"]
        upd, sig_prev, m_prev = h["median_update_norm"], h["median_sigma"], m
    return history, states


def controller_v3_path():
    tag = f"__{CFG['TAG']}" if CFG.get("TAG") else ""
    return os.path.join(CFG["RESULTS_DIR"], f"controller_v3_shared{tag}.pt")


def _practice_episode(ep_seed, eps_budget, C, policy, XA, yA, val_B, q_real, label):
    set_seed(ep_seed)
    parts = dirichlet_split(yA, NUM_CLIENTS, ALPHA, np.random.default_rng(ep_seed), CFG["PROXY_MIN_PER_CLIENT"])
    w = reward_weights()
    mk = lambda: SmallDDQN(input_dim, action_dim, device, w, gamma=0.0, lr=CFG["LR"])
    g = mk()
    proxies = [DPClient(mk(), XA[ci], yA[ci], device, model_name=f"Proxy{k+1}", batch_size=CFG["BATCH"],
                        clip_norm=C, delta=CFG["DELTA"]) for k, ci in enumerate(parts)]
    R, STEPS = CFG["ROUNDS"], CFG["LOCAL_STEPS"]
    pacers = [RDPPacer(q_real[k], eps_budget, CFG["DELTA"], R, STEPS) for k in range(len(proxies))]
    hist, _ = paced_clip_episode(label, g, proxies, pacers, policy, val_B, rounds=R, steps=STEPS, mu=CFG["MU"],
                                 rng=np.random.default_rng(ep_seed), C_base=C, eps_budget=eps_budget,
                                 fast=make_fast(proxies))
    return episode_score(hist), hist


def meta_train_controller_v3(C):
    """Meta-train ONE ADP-Meta v3 controller for all eps in CFG['META_EPS'] (cached)."""
    path = controller_v3_path()
    ctrl = ADPMetaV3(CFG["PACE_ACTIONS"], CFG["CLIP_ACTIONS"], lr=CFG["ADP_LR"], entropy_coef=CFG["ADP_ENTROPY"])
    if os.path.exists(path):
        ctrl.load_state_dict(torch.load(path)); print(f"Loaded meta-trained controller {path}")
        return ctrl
    torch.manual_seed(SPLIT_SEED)
    ctrl = ADPMetaV3(CFG["PACE_ACTIONS"], CFG["CLIP_ACTIONS"], lr=CFG["ADP_LR"], entropy_coef=CFG["ADP_ENTROPY"])
    ia, ib = val_halves()
    XA, yA = X_val[ia], y_val[ia]
    val_B = TensorDataset(torch.from_numpy(X_val[ib]), torch.from_numpy(y_val[ib]).long())
    q_real = [min(1.0, CFG["BATCH"] / len(ci)) for ci in dp_client_indices]
    pool = [20_000 + i for i in range(CFG["TWIN_POOL"])]
    twins, log, t0 = {}, [], time.time()
    fixed = lambda s: (1.0, 1.0)
    for e in range(CFG["ADP_META_EPISODES"]):
        eps_b = CFG["META_EPS"][e % len(CFG["META_EPS"])]
        es = pool[(e // len(CFG["META_EPS"])) % len(pool)]
        if (es, eps_b) not in twins:
            twins[(es, eps_b)] = _practice_episode(es, eps_b, C, fixed, XA, yA, val_B, q_real,
                                                   f"twin s{es} eps{eps_b:g}")[0]
        score, hist = _practice_episode(es, eps_b, C, lambda s: ctrl.act(s), XA, yA, val_B, q_real,
                                        f"meta ep{e+1} eps{eps_b:g}")
        adv = score - twins[(es, eps_b)]
        ctrl.finish_episode(adv)
        log.append({"episode": e + 1, "eps": eps_b, "seed": es, "score": score, "twin": twins[(es, eps_b)],
                    "advantage": adv, "actions": [(h["action_k"], h["clip_mult"]) for h in hist]})
        print(f"=== meta-episode {e+1}/{CFG['ADP_META_EPISODES']} eps={eps_b:g} | score={100*score:.1f} "
              f"twin={100*twins[(es, eps_b)]:.1f} adv={100*adv:+.1f} | {(time.time()-t0)/60:.0f} min")
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    torch.save(ctrl.state_dict(), path)
    with open(path.replace(".pt", "_meta_log.json"), "w") as f:
        json.dump({"episodes": log, "minutes": (time.time() - t0) / 60, "val_halves_seed": SPLIT_SEED,
                   "cfg": cfg_json()}, f, indent=2)
    return ctrl


def _run_paced_clip(name, method, seed, eps_budget, C, g, clients, parts, rng, mu, fast=None):
    ctrl = meta_train_controller_v3(C)
    set_seed(seed); rng = np.random.default_rng(seed)        # meta-training must not shift the run's RNG
    R, STEPS, delta = CFG["ROUNDS"], CFG["LOCAL_STEPS"], CFG["DELTA"]
    pacers = [RDPPacer(c.q, eps_budget, delta, R, STEPS) for c in clients]
    print(f"[{name} | seed {seed}] ADP-Meta v3 (frozen, greedy) | C_base={C:.3f} | mu={mu} | "
          f"sigma0 median={np.median([p.sigma0 for p in pacers]):.3f}")
    t0 = time.time()
    history, states = paced_clip_episode(name, g, clients, pacers, lambda s: ctrl.act(s, greedy=True), val_dataset,
                                         rounds=R, steps=STEPS, mu=mu, rng=rng, C_base=C, eps_budget=eps_budget,
                                         fast=fast, check_budget=True, keep_states=True)
    return _save(name, seed, history, states, "small", {
        "method": method, "private": True, "eps_budget": eps_budget,
        "final_epsilon": history[-1]["epsilon"] if history else 0.0, "delta": delta, "clip_norm": C,
        "mu": mu, "adp_version": "v3-pacing-clip", "adp_mode": "greedy", "controller_file": controller_v3_path(),
        "sigma0_per_client": [p.sigma0 for p in pacers], "rounds_run": len(history),
        "minutes": (time.time() - t0) / 60})


In [ ]:
# =====================================================================================
# BLOCK 18a (v4 - C0): Experiment definitions - ONE runner for every method, VALIDATION ONLY
#   Methods (same seeds for all -> paired comparison):
#     fedavg_np / fedprox_np  : non-private, SAME pipeline as DP (all train data, small DDQN, gamma 0,
#                               Poisson batches, 20 rounds x 400 steps) but no clipping and no noise
#     dp_clip_nonoise         : DP pipeline with clipping but sigma = 0 (diagnostic: cost of clipping alone)
#     dp_fixed                : record-level DP-SGD, constant noise calibrated to the budget over all rounds
#     dp_adpmeta              : record-level DP-SGD, ADP-Meta controller (Block 17) picks the noise each round
#     fedavg_ch5 / fedprox_ch5: Chapter 5 non-private pipeline (resampled data, 42.6k DDQN, gamma 0.1) - reference
#   Model selection (fixed BEFORE any new test result, applied to EVERY method):
#     highest validation macro-F1 among rounds with validation FAR <= FAR_CAP;
#     if no round meets the cap -> round with the lowest validation FAR, and cap_met = False is recorded.
#   The test set is NOT touched here. Block 19 evaluates the selected models on test, once, at the end.
# Seeds: train/val/test split, scaler and feature selection stay fixed (SEED = 42, Blocks 8/8b).
#        run_seed drives model init, Dirichlet client split, Poisson sampling, client order,
#        ADP-Meta policy init and (Chapter 5 runs) the resampling.
# =====================================================================================
import copy, time, json, os, gc, traceback
from opacus.accountants.utils import get_noise_multiplier

SPLIT_SEED = SEED                      # 42 - fixed data split / scaler / feature selection
CFG = dict(
    ROUNDS=20, LOCAL_STEPS=400, BATCH=256, LR=1e-3, CLIENT_FRAC=1.0, MU=0.01, DELTA=1e-5,
    FAR_CAP=0.10,
    LEVEL_MULTS=(0.8, 0.9, 1.0, 1.2, 1.5), LAMBDA=1.0,         # ADP-Meta v1 (legacy, ADP_ACTIONS="mult")
    ADP_ACTIONS="mult",                                          # "mult" = v1 (C0) | "pacing" = v2 (C5, Block 17b)
    PACE_ACTIONS=(0.5, 0.75, 1.0, 1.5, 2.0), ADP_LR=1e-2, ADP_ENTROPY=0.01,
    ADP_META_EPISODES=0,                                         # C6: >0 -> meta-train on validation, then frozen
    PROXY_MIN_PER_CLIENT=50,
    TAG="",                                                      # appended to run names: one tag per ablation
    # Sweep-1 knobs (defaults = previous behaviour)
    WEIGHTS_FROM="train",      # C1: reward class weights from pooled "train" counts or server-held "val" counts
    REWARD_POWER=0.5,          # C4: weights = balanced_weight ** power (0.5 = square root, as before)
    REWARD_FA=0.0,             # C4: extra penalty when a NORMAL flow is flagged as an attack (false alarm)
    REWARD_MISS=0.0,           # C4: extra penalty when an ATTACK is called normal (missed attack)
    CLIP_MULT=1.0,             # C3: C = CLIP_MULT x median per-sample grad norm on validation
    CH5_ROUNDS=50, CH5_FRAC=0.8, CH5_LOCAL_EPOCHS=2, CH5_OVERSAMPLE_CAP=10,
    RESULTS_DIR="results",
    FAST=True,
    # ADP-Meta v3 (Block 17c, ADP_ACTIONS = "pacing_clip")
    CLIP_ACTIONS=(0.5, 1.0, 2.0),   # clip multiplier on C_base chosen each round
    META_EPS=(8.0, 3.0, 1.0),       # one shared controller is meta-trained across these budgets
    TWIN_POOL=4,                    # practice seeds; each has one cached fixed-schedule twin per eps                 # Block 16b: train all clients in parallel on the GPU (same algorithm, much faster)
)
_DEFAULT_CFG = dict(CFG)               # Block 18b resets CFG to these defaults before applying a stage

# ------------------------------ smaller DDQN network for private training (D10) ------------------------------
class SmallQNetwork(nn.Module):
    def __init__(self, input_dim, action_dim):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.ReLU(),
            nn.Linear(64, 32),        nn.LayerNorm(32), nn.ReLU())
        self.q_head = nn.Linear(32, action_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)
    def forward(self, x, mode="q_value"):
        return self.q_head(self.feature_extractor(x))

NORMAL_IDX = class_names.index("normal")

class SmallDDQN(DDQN):                       # identical DDQN logic (reward, TD update, target net), smaller network
    def __init__(self, *args, lr=1e-3, **kwargs):
        super().__init__(*args, lr=lr, **kwargs)
        self.online_net = SmallQNetwork(input_dim, action_dim).to(self.device)
        self.target_net = SmallQNetwork(input_dim, action_dim).to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict()); self.target_net.eval()
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)

    def reward(self, actions, labels):
        """+w[y] if correct, -w[y] if wrong (as before), plus optional cost-sensitive IDS penalties (C4).
        Still a pure reward: the detector is trained only with the TD loss on this reward."""
        w = self.reward_w[labels]
        r = torch.where(actions == labels, w, -w)
        wrong = actions != labels
        if CFG["REWARD_FA"]:
            r = r - CFG["REWARD_FA"] * (wrong & (labels == NORMAL_IDX)).float()
        if CFG["REWARD_MISS"]:
            r = r - CFG["REWARD_MISS"] * (wrong & (labels != NORMAL_IDX) & (actions == NORMAL_IDX)).float()
        return r

def reward_weights():
    """Class weights for the DP-pipeline reward. 'val' uses server-held validation counts, so no statistic of the
    private training data is used outside DP (C1)."""
    yy = {"train": y_dp, "val": y_val}[CFG["WEIGHTS_FROM"]]
    cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=yy) ** CFG["REWARD_POWER"]
    return (cw / cw.mean()).astype(np.float32)

ARCH = {"small": SmallDDQN, "large": DDQN}

# ------------------------------ helpers ------------------------------
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def prf_from_cm(cm):
    cm = np.array(cm, dtype=float); tp = np.diag(cm)
    p  = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    r  = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    return p, r, f1, cm.sum(1)

def eval_metrics(agent, dataset):
    """Accuracy, IDS FAR/DR, macro P/R/F1 and per-class F1 of `agent` on `dataset`."""
    m = evaluate_global(agent, dataset)
    ids = m.ids_metrics()
    P, R, F1, sup = prf_from_cm(m.cm)
    return {"Accuracy": ids["Accuracy"], "FAR": ids["IDS_FAR"], "DR": ids["IDS_DR"],
            "Macro_P": float(P.mean()), "Macro_R": float(R.mean()), "Macro_F1": float(F1.mean()),
            "Weighted_F1": float((F1 * sup / sup.sum()).sum()),
            "F1_per_class": {n: float(f) for n, f in zip(class_names, F1)}, "cm": m.cm.tolist()}

def select_best_round(history, far_cap):
    """Index into history. Rule: max val macro-F1 among rounds with val FAR <= far_cap (earliest on ties);
    if none qualifies -> lowest val FAR (higher macro-F1, then earliest, on ties). Returns (index, cap_met)."""
    assert history, "empty history"
    ok = [i for i, h in enumerate(history) if h["FAR"] <= far_cap]
    if ok:
        return max(ok, key=lambda i: (history[i]["Macro_F1"], -i)), True
    return min(range(len(history)), key=lambda i: (history[i]["FAR"], -history[i]["Macro_F1"], i)), False

def per_sample_grad_norms(agent, X, y, n=1024):          # used only to pick C (server-held VALIDATION data)
    net = agent.online_net; net.train()
    idx = np.random.default_rng(SPLIT_SEED).choice(len(y), size=min(n, len(y)), replace=False)
    x  = torch.as_tensor(X[idx], device=device); yy = torch.as_tensor(y[idx], device=device)
    a  = agent.act(x); r = agent.reward(a, yy)
    params  = {k: v.detach() for k, v in net.named_parameters()}
    buffers = {k: v.detach() for k, v in net.named_buffers()}
    def loss_fn(p, b, xi, ai, ri):
        qv = functional_call(net, (p, b), (xi.unsqueeze(0),)).squeeze(0)
        return F.smooth_l1_loss(qv.gather(0, ai.unsqueeze(0)).squeeze(0), ri)
    g = vmap(grad(loss_fn), in_dims=(None, None, 0, 0, 0), randomness="different")(params, buffers, x, a, r)
    return torch.sqrt(sum(gi.reshape(len(idx), -1).pow(2).sum(1) for gi in g.values())).cpu().numpy()

def dp_clip_norm():
    """C = median per-sample grad norm at init on VALIDATION data. Fixed seed -> same C for every method and seed."""
    set_seed(SPLIT_SEED)
    ag = SmallDDQN(input_dim, action_dim, device, reward_weights(), gamma=0.0, lr=CFG["LR"])
    return CFG["CLIP_MULT"] * float(np.median(per_sample_grad_norms(ag, X_val, y_val)))

def run_name(method, eps_budget=None):
    base = method if eps_budget is None else f"{method}_eps{eps_budget:g}"
    return f"{base}__{CFG['TAG']}" if CFG.get("TAG") else base

def cfg_json():
    return {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}

def result_path(name, seed, suffix=".json"):
    return os.path.join(CFG["RESULTS_DIR"], f"{name}__seed{seed}{suffix}")

def _save(name, seed, history, states, arch, extra):
    bi, cap_met = select_best_round(history, CFG["FAR_CAP"])
    os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
    torch.save(states[bi], result_path(name, seed, "_best.pt"))
    out = {"run": name, "seed": seed, "split_seed": SPLIT_SEED, "arch": arch,
           "selection_rule": f"max val Macro_F1 s.t. val FAR <= {CFG['FAR_CAP']}; else min val FAR",
           "best_round": history[bi]["round"], "cap_met": cap_met, "val_at_best": history[bi],
           "history": [{k: v for k, v in h.items() if k != "cm"} for h in history],
           "cfg": cfg_json(),
           "features": list(selected_features), **extra}
    with open(result_path(name, seed), "w") as f:
        json.dump(out, f, indent=2)
    h = history[bi]
    print(f"[{name} | seed {seed}] selected round {h['round']} (cap met: {cap_met}) | VAL Acc={100*h['Accuracy']:.2f}% "
          f"MacroF1={100*h['Macro_F1']:.1f} FAR={100*h['FAR']:.2f}% DR={100*h['DR']:.2f}%")
    return out

def _log_round(name, r, h, extra=""):
    print(f"[{name}] Round {r:02d}{extra} | VAL Acc={100*h['Accuracy']:.2f}% F1={100*h['Macro_F1']:.1f} "
          f"FAR={100*h['FAR']:.2f}% DR={100*h['DR']:.2f}%")
    print("          per-class F1: " + " | ".join(f"{n}={100*f:.0f}" for n, f in h["F1_per_class"].items()))

# ------------------------------ DP-pipeline runner (fedavg_np, fedprox_np, dp_*) ------------------------------
METHODS_DP_PIPELINE = {
    #  method            : (mu,          clip,   private, adaptive)
    "fedavg_np"          : (0.0,         False,  False,   False),
    "fedprox_np"         : ("MU",        False,  False,   False),
    "dp_clip_nonoise"    : ("MU",        True,   False,   False),
    "dp_fixed"           : ("MU",        True,   True,    False),
    "dp_adpmeta"         : ("MU",        True,   True,    True),
}

def run_dp_pipeline(method, seed, eps_budget=None, C=None):
    mu, use_clip, private, adaptive = METHODS_DP_PIPELINE[method]
    mu = CFG["MU"] if mu == "MU" else mu
    assert private == (eps_budget is not None), f"{method}: eps_budget must be given iff the method is private"
    name = run_name(method, eps_budget)
    R, STEPS, delta = CFG["ROUNDS"], CFG["LOCAL_STEPS"], CFG["DELTA"]

    set_seed(seed)
    rng = np.random.default_rng(seed)
    parts = dirichlet_split(y_dp, NUM_CLIENTS, ALPHA, np.random.default_rng(seed), MIN_PER_CLIENT)
    _a = np.concatenate(parts)
    assert len(_a) == len(y_dp) and len(np.unique(_a)) == len(y_dp), "A record is in more than one client"

    w = reward_weights()
    mk = lambda: SmallDDQN(input_dim, action_dim, device, w, gamma=0.0, lr=CFG["LR"])
    g = mk()
    clients = [DPClient(mk(), X_dp[ci], y_dp[ci], device, model_name=f"Client{k+1}", batch_size=CFG["BATCH"],
                        clip_norm=(C if use_clip else None), delta=delta) for k, ci in enumerate(parts)]
    if private:
        base = [get_noise_multiplier(target_epsilon=eps_budget, target_delta=delta, sample_rate=c.q,
                                     steps=R * STEPS, accountant="rdp") for c in clients]
    else:
        base = [0.0] * len(clients)
    fast = make_fast(clients)
    if adaptive and CFG["ADP_ACTIONS"] == "pacing_clip":           # ADP-Meta v3 (Block 17c)
        return _run_paced_clip(name, method, seed, eps_budget, C, g, clients, parts, rng, mu, fast)
    if adaptive and CFG["ADP_ACTIONS"] == "pacing":                # ADP-Meta v2 (C5/C6, Block 17b)
        return _run_paced(name, method, seed, eps_budget, C, g, clients, parts, rng, mu, fast)
    ctrl = ADPMetaController(list(CFG["LEVEL_MULTS"]), device=device) if adaptive else None
    print(f"[{name} | seed {seed}] clip={C if use_clip else 'none'} | mu={mu} | "
          f"sigma median={np.median(base):.3f} | client sizes={[len(p) for p in parts]}")

    history, states, ctrl_hist = [], [], []
    acc_prev, dacc_prev, eps_prev = eval_metrics(g, val_dataset)["Accuracy"], 0.0, 0.0
    t0 = time.time()
    for r in range(1, R + 1):
        mult, forced = 1.0, False
        if adaptive:
            mult = ctrl.select([r / R, acc_prev, dacc_prev, eps_prev / eps_budget])[1]
        active = rng.choice(len(clients), size=max(1, math.ceil(CFG["CLIENT_FRAC"] * len(clients))), replace=False)

        if private:                                         # pre-round budget check: nobody may exceed the budget
            over = lambda m: max(clients[i].projected_epsilon(base[i] * m, STEPS) for i in active) > eps_budget
            if over(mult):
                if adaptive and not over(max(CFG["LEVEL_MULTS"])):
                    mult, forced = max(CFG["LEVEL_MULTS"]), True
                else:
                    print(f"[{name}] Round {r}: privacy budget would be exceeded - stopping before spending.")
                    break

        gp = g.get_fl_parameters()
        params, sizes, _ = fit_clients(clients, active, gp, {int(i): base[i] * mult for i in active}, STEPS, mu, fast)
        g.set_fl_parameters(weighted_average(params, sizes))

        h = eval_metrics(g, val_dataset)
        eps_now = max(c.epsilon_spent() for c in clients) if private else None
        if private:
            assert eps_now <= eps_budget + 1e-9, "DP guard: budget exceeded"
        if adaptive:
            dacc = h["Accuracy"] - acc_prev
            reward = dacc - CFG["LAMBDA"] * (eps_now - eps_prev) / eps_budget
            if not forced:
                ctrl.update(reward, r)
            ctrl_hist.append({"round": r, "mult": mult, "forced": forced, "reward": reward})
            dacc_prev = dacc
        h.update({"round": r, "noise_mult": mult, "forced": forced, "epsilon": eps_now,
                  "median_sigma": float(np.median(base)) * mult})
        history.append(h); states.append([t.clone() for t in g.get_fl_parameters()])
        _log_round(name, r, h, f" | mult={mult:.2f}{'*' if forced else ''} | eps={eps_now if eps_now is None else round(eps_now, 3)}")
        acc_prev = h["Accuracy"]
        if private:
            eps_prev = eps_now

    return _save(name, seed, history, states, "small", {
        "method": method, "private": private, "eps_budget": eps_budget,
        "final_epsilon": eps_prev if private else None,   # TOTAL spent over all rounds (server saw every update)
        "delta": delta if private else None, "clip_norm": C if use_clip else None, "mu": mu,
        "base_sigma_per_client": base, "rounds_run": len(history), "minutes": (time.time() - t0) / 60,
        "controller": ctrl_hist})

def make_fast(clients):
    return FastFederation(clients) if CFG["FAST"] and CFG["CLIENT_FRAC"] >= 1.0 else None

def _run_paced(name, method, seed, eps_budget, C, g, clients, parts, rng, mu, fast=None):
    R, STEPS, delta = CFG["ROUNDS"], CFG["LOCAL_STEPS"], CFG["DELTA"]
    meta = CFG["ADP_META_EPISODES"] > 0
    if meta:
        ctrl, mode = meta_train_controller(eps_budget, C), "greedy"           # frozen, trained on validation only
        set_seed(seed); rng = np.random.default_rng(seed)                    # meta-training must not shift the run's RNG
    else:
        ctrl, mode = ADPMetaV2(CFG["PACE_ACTIONS"], lr=CFG["ADP_LR"], entropy_coef=CFG["ADP_ENTROPY"],
                               device=device), "sample_online"
    pacers = [RDPPacer(c.q, eps_budget, delta, R, STEPS) for c in clients]
    print(f"[{name} | seed {seed}] ADP-Meta v2 ({mode}) | clip={C} | mu={mu} | "
          f"sigma0 median={np.median([p.sigma0 for p in pacers]):.3f} | client sizes={[len(p) for p in parts]}")
    t0 = time.time()
    history, states, _ = paced_episode(name, g, clients, pacers, ctrl, val_dataset, rounds=R, steps=STEPS, mu=mu,
                                       rng=rng, mode=mode, eps_budget=eps_budget, keep_states=True, fast=fast)
    return _save(name, seed, history, states, "small", {
        "method": method, "private": True, "eps_budget": eps_budget,
        "final_epsilon": history[-1]["epsilon"] if history else 0.0,
        "delta": delta, "clip_norm": C, "mu": mu, "adp_version": "v2-pacing", "adp_mode": mode,
        "controller_file": controller_path(eps_budget) if meta else None,
        "sigma0_per_client": [p.sigma0 for p in pacers], "rounds_run": len(history),
        "minutes": (time.time() - t0) / 60})

# ------------------------------ Chapter 5 non-private reference (fedavg_ch5, fedprox_ch5) ------------------------------
def ch5_clients(seed):
    """Resampling + Dirichlet split + loaders exactly as Blocks 9/10, but seeded with run_seed."""
    counts = Counter(y_train); tgt = int(np.median(list(counts.values())))
    over  = {c: min(tgt, n * CFG["CH5_OVERSAMPLE_CAP"]) for c, n in counts.items() if n < tgt}
    under = {c: tgt for c, n in counts.items() if n > tgt}
    Xr, yr = RandomOverSampler(sampling_strategy=over, random_state=seed).fit_resample(X_train, y_train)
    Xr, yr = RandomUnderSampler(sampling_strategy=under, random_state=seed).fit_resample(Xr, yr)
    Xr, yr = Xr.astype(np.float32), yr.astype(np.int64)
    cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=yr)
    w = (np.sqrt(cw) / np.sqrt(cw).mean()).astype(np.float32)
    ds = TensorDataset(torch.from_numpy(Xr), torch.from_numpy(yr).long())
    parts = dirichlet_split(yr, NUM_CLIENTS, ALPHA, np.random.default_rng(seed), MIN_PER_CLIENT)
    loaders = [DataLoader(Subset(ds, ci.tolist()), batch_size=32, shuffle=True,
                          generator=torch.Generator().manual_seed(seed + k)) for k, ci in enumerate(parts)]
    return loaders, w

def run_ch5(method, seed):
    assert DP_ENABLED is False, "Chapter 5 runs need DP_ENABLED = False (Block 13)"
    mu = {"fedavg_ch5": 0.0, "fedprox_ch5": CFG["MU"]}[method]
    set_seed(seed)
    rng = np.random.default_rng(seed)
    loaders, w = ch5_clients(seed)
    g = DDQN(input_dim, action_dim, device, w)
    clients = [FLClient(DDQN(input_dim, action_dim, device, w), loaders[k], device, model_name=f"Client{k+1}")
               for k in range(NUM_CLIENTS)]
    history, states, t0 = [], [], time.time()
    for r in range(1, CFG["CH5_ROUNDS"] + 1):
        active = rng.choice(NUM_CLIENTS, size=max(1, math.ceil(CFG["CH5_FRAC"] * NUM_CLIENTS)), replace=False)
        gp = g.get_fl_parameters()
        params, sizes = [], []
        for i in active:
            p, n, _ = clients[i].fit(gp, {"local_epochs": CFG["CH5_LOCAL_EPOCHS"], "mu": mu})
            params.append(p); sizes.append(n)
        g.set_fl_parameters(weighted_average(params, sizes))
        h = eval_metrics(g, val_dataset); h["round"] = r
        history.append(h); states.append([t.clone() for t in g.get_fl_parameters()])
        _log_round(method, r, h)
    return _save(method, seed, history, states, "large", {
        "method": method, "private": False, "eps_budget": None, "final_epsilon": None, "mu": mu,
        "rounds_run": len(history), "minutes": (time.time() - t0) / 60, "class_weights": w.tolist()})

# ------------------------------ batch runner ------------------------------
def run_all(experiments, seeds):
    """experiments: list of (method, eps_budget or None). Finished (method, eps, seed) runs are skipped."""
    C = None
    T0 = time.time()
    for seed in seeds:
        for method, eps in experiments:
            name = run_name(method, eps)
            if os.path.exists(result_path(name, seed)):
                with open(result_path(name, seed)) as f:
                    old = json.load(f)["cfg"]
                diff = {k for k in set(old) | set(cfg_json()) if k != "RESULTS_DIR" and old.get(k) != cfg_json().get(k)}
                if diff:
                    raise ValueError(f"{name} seed {seed} exists with a DIFFERENT config ({sorted(diff)}). "
                                     f"Set CFG['TAG'] to a new name for this ablation.")
                print(f">>> SKIP {name} seed {seed} (done)"); continue
            print(f"\n{'='*70}\n>>> START {name} | seed {seed} | elapsed {(time.time()-T0)/60:.0f} min\n{'='*70}")
            try:
                if method in METHODS_DP_PIPELINE:
                    if C is None:
                        C = dp_clip_norm(); print(f"DP clip norm C (median per-sample grad norm, validation): {C:.3f}")
                    run_dp_pipeline(method, seed, eps, C)
                else:
                    assert eps is None, f"{method} is non-private"
                    run_ch5(method, seed)
            except Exception:
                print(f"!!! {name} seed {seed} FAILED:"); traceback.print_exc()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

def summarize_val(results_dir=None):
    """Validation-only summary of every finished run (mean +/- std over seeds). Never reads test results."""
    rows = []
    d = results_dir or CFG["RESULTS_DIR"]
    for fn in sorted(os.listdir(d)) if os.path.isdir(d) else []:
        if fn.endswith(".json") and "__seed" in fn and not fn.endswith("_TEST.json"):
            with open(os.path.join(d, fn)) as f:
                r = json.load(f)
            v = r["val_at_best"]
            rows.append({"run": r["run"], "seed": r["seed"], "best_round": r["best_round"], "cap_met": r["cap_met"],
                         "Acc": 100 * v["Accuracy"], "MacroF1": 100 * v["Macro_F1"], "FAR": 100 * v["FAR"],
                         "DR": 100 * v["DR"], "eps": r.get("final_epsilon"),
                         **{f"F1_{k}": 100 * x for k, x in v["F1_per_class"].items()}})
    return pd.DataFrame(rows)


In [ ]:
# BLOCK 18b (v4): Run experiments (VALIDATION only). Finished runs are skipped, so re-running resumes.
# Default STAGE = "v3_val": ADP-Meta v3 (noise pacing + adaptive clip, twin-baseline meta-training, one shared
# controller for eps 8/3/1) vs fixed noise on the frozen settings, seed 0, eps 8 and 1. VALIDATION only.
# Success rule written BEFORE running (CHANGELOG): ADP-Meta v3 must beat fixed noise by >= 2 val macro-F1 with
# FAR <= 10% at an eps; then the 3-seed stage "final_v3" decides (mean better AND better on >= 2 of 3 seeds).
# The test set (Block 19) is used only once, after "final_v3".
# Other choice: STAGE = "eps16" -> one more point on the privacy-utility curve (weaker privacy, eps = 16):
#   ADP-Meta v2 (frozen final settings, meta-trained for eps 16) vs fixed noise, seeds 0/1/2 (~1.5-2 h).
STAGE = "none"

DP8 = [("dp_fixed", 8.0)]
C0 = [("", {}, [0], [("fedavg_np", None), ("fedprox_np", None), ("dp_clip_nonoise", None),
                     ("dp_fixed", 8.0), ("dp_adpmeta", 8.0)])]
SWEEP1 = [  # (TAG, one change, seeds, runs)
    ("W_val",      dict(WEIGHTS_FROM="val"),                [0], DP8),   # C1 weights from validation counts
    ("B1024_S100", dict(BATCH=1024, LOCAL_STEPS=100),       [0], DP8),   # C3 bigger batches, fewer steps
    ("B1024_S200", dict(BATCH=1024, LOCAL_STEPS=200),       [0], DP8),
    ("B2048_S100", dict(BATCH=2048, LOCAL_STEPS=100),       [0], DP8),
    ("CLIPx0.5",   dict(CLIP_MULT=0.5),                     [0], DP8),   # C3 clip norm
    ("CLIPx2",     dict(CLIP_MULT=2.0),                     [0], DP8),
    ("LR3e-3",     dict(LR=3e-3),                           [0], DP8),   # C3 learning rate
    ("FA0.5",      dict(REWARD_FA=0.5),                     [0], DP8),   # C4 false-alarm penalty
    ("MISS0.5",    dict(REWARD_MISS=0.5),                   [0], DP8),   # C4 missed-attack penalty
    ("POW1.0",     dict(REWARD_POWER=1.0),                  [0], DP8),   # C4 fully balanced weights
    ("MU0",        dict(MU=0.0),                            [0], DP8),   # C11 FedProx mu
    ("MU0.1",      dict(MU=0.1),                            [0], DP8),
]
MAIN = [("",   {},                        [0], [("dp_adpmeta", 8.0)]),     # before
        ("C5", dict(ADP_ACTIONS="pacing"),   [0], [("dp_adpmeta", 8.0)])]     # after: ADP-Meta v2
BASE2 = dict(CLIP_MULT=0.5, REWARD_FA=0.5)                                  # Sweep-1 winners combined
NEXT = [
    ("S2base",    BASE2,                                              [0], [("dp_fixed", 8.0)]),   # reference
    ("S2_ADPv2",  BASE2 | dict(ADP_ACTIONS="pacing"),                 [0], [("dp_adpmeta", 8.0)]), # ADP-Meta online
    ("S2_ADPmeta", BASE2 | dict(ADP_ACTIONS="pacing", ADP_META_EPISODES=10), [0], [("dp_adpmeta", 8.0)]),  # + meta-learning
    ("S2_CLIP0.25", BASE2 | dict(CLIP_MULT=0.25),                     [0], [("dp_fixed", 8.0)]),   # refinements (one change each)
    ("S2_FA1.0",  BASE2 | dict(REWARD_FA=1.0),                        [0], [("dp_fixed", 8.0)]),
    ("S2_MISS0.5", BASE2 | dict(REWARD_MISS=0.5),                     [0], [("dp_fixed", 8.0)]),
    ("S2_B1024",  BASE2 | dict(BATCH=1024, LOCAL_STEPS=200),          [0], [("dp_fixed", 8.0)]),
]
FINAL_CFG = BASE2 | dict(ADP_ACTIONS="pacing", ADP_META_EPISODES=10)        # frozen (CHANGELOG, Sweep 2)
V3_CFG = BASE2 | dict(ADP_ACTIONS="pacing_clip", ADP_META_EPISODES=24)      # ADP-Meta v3 (Block 17c)
STAGES = {
    "none": [],                                  # run nothing here (e.g. the Torre-CNN comparison notebook)
    "v3_val": [("v3", V3_CFG, [0], [("dp_fixed", 8.0), ("dp_adpmeta", 8.0), ("dp_fixed", 1.0), ("dp_adpmeta", 1.0)])],
    "final_v3": [("v3", V3_CFG, [0, 1, 2], [
        ("fedavg_np", None), ("fedprox_np", None),
        ("dp_fixed", 8.0), ("dp_adpmeta", 8.0), ("dp_fixed", 3.0), ("dp_adpmeta", 3.0),
        ("dp_fixed", 1.0), ("dp_adpmeta", 1.0)])],
    "eps16": [("final", FINAL_CFG, [0, 1, 2], [("dp_fixed", 16.0), ("dp_adpmeta", 16.0)])],
    "next": NEXT,
    "main": MAIN,
    "auto": C0 + SWEEP1,
    "C0_baseline": C0,
    "sweep1": SWEEP1,
    # ADP-Meta v2 (run after the fixed-noise settings are chosen; add the accepted settings to cfg)
    "C5_pacing_online": [("C5", dict(ADP_ACTIONS="pacing"), [0], [("dp_adpmeta", 8.0)])],
    "C6_meta": [("C6", dict(ADP_ACTIONS="pacing", ADP_META_EPISODES=10), [0], [("dp_adpmeta", 8.0)])],
    # FINAL: frozen settings, 3 paired seeds, eps 1/3/8 (Chapter 5 reference rows are a separate optional stage)
    "final": [("final", FINAL_CFG, [0, 1, 2], [
        ("fedavg_np", None), ("fedprox_np", None),
        ("dp_fixed", 8.0), ("dp_adpmeta", 8.0), ("dp_fixed", 3.0), ("dp_adpmeta", 3.0),
        ("dp_fixed", 1.0), ("dp_adpmeta", 1.0)])],
    "ch5_reference": [("ch5", {}, [0, 1, 2], [("fedavg_ch5", None), ("fedprox_ch5", None)])],
}

for tag, over, seeds, exps in STAGES[STAGE]:
    CFG.clear(); CFG.update(_DEFAULT_CFG); CFG.update(over); CFG["TAG"] = tag
    print(f"\n##### Stage {STAGE} | variant '{tag or 'baseline'}' | changes: {over or 'none'}")
    run_all(exps, seeds)
CFG.clear(); CFG.update(_DEFAULT_CFG)


In [ ]:
# BLOCK 18c (v4): Validation summary - SEND THIS OUTPUT BACK (no test numbers are involved)
REF = "dp_fixed_eps8__S2base"      # reference run for the "vs reference" table (fixed noise, same settings)
vdf = summarize_val()
if vdf.empty:
    print("No finished runs in", CFG["RESULTS_DIR"])
else:
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
    main = ["run", "seed", "best_round", "cap_met", "Acc", "MacroF1", "FAR", "DR", "eps"]
    print("VALIDATION - selected round per run\n", vdf[main].round(2).to_string(index=False))
    print("\nVALIDATION - per-class F1 at the selected round\n",
          vdf[["run", "seed"] + [c for c in vdf.columns if c.startswith("F1_")]].round(1).to_string(index=False))

    # Before/after: every tagged variant vs its untagged baseline run (same method, eps and seed)
    rows = []
    for _, c in vdf[vdf["run"].str.contains("__")].iterrows():
        base_name, variant = c["run"].split("__", 1)
        bb = vdf[(vdf["run"] == base_name) & (vdf["seed"] == c["seed"])]
        if bb.empty:
            continue
        bb = bb.iloc[0]
        dF1, dFAR = c["MacroF1"] - bb["MacroF1"], c["FAR"] - bb["FAR"]
        rows.append({"baseline": base_name, "variant": variant, "seed": c["seed"],
                     "base_MacroF1": bb["MacroF1"], "MacroF1": c["MacroF1"], "dMacroF1": dF1,
                     "base_Acc": bb["Acc"], "Acc": c["Acc"], "dAcc": c["Acc"] - bb["Acc"],
                     "base_FAR": bb["FAR"], "FAR": c["FAR"], "dFAR": dFAR, "cap_met": c["cap_met"],
                     "verdict": "IMPROVES" if (dF1 >= 2.0 and dFAR <= 1.0) else "no"})
    if rows:
        cmp_df = pd.DataFrame(rows).sort_values(["baseline", "dMacroF1"], ascending=[True, False])
        print("\nBEFORE / AFTER (validation) - IMPROVES = +2 macro-F1 or more and FAR not worse by more than 1 point\n",
              cmp_df.round(2).to_string(index=False))
        cmp_df.to_csv(os.path.join(CFG["RESULTS_DIR"], "before_after.csv"), index=False)

    ref = vdf[vdf["run"] == REF]
    if len(ref):
        rr = ref.iloc[0]
        t = vdf[vdf["seed"] == rr["seed"]][["run", "best_round", "cap_met", "Acc", "MacroF1", "FAR", "DR", "eps"]].copy()
        t["dMacroF1"], t["dAcc"], t["dFAR"] = t["MacroF1"] - rr["MacroF1"], t["Acc"] - rr["Acc"], t["FAR"] - rr["FAR"]
        t["verdict"] = np.where((t["dMacroF1"] >= 2.0) & (t["dFAR"] <= 1.0), "IMPROVES", "no")
        t.loc[t["run"] == REF, "verdict"] = "REFERENCE"
        print(f"\nVS REFERENCE {REF} (validation)\n", t.sort_values("MacroF1", ascending=False).round(2).to_string(index=False))
        t.to_csv(os.path.join(CFG["RESULTS_DIR"], "vs_reference.csv"), index=False)

    # Paired ADP-Meta vs fixed noise (same tag, eps and seed)
    pairs = []
    for _, c in vdf[vdf["run"].str.startswith("dp_adpmeta_eps")].iterrows():
        f = vdf[(vdf["run"] == c["run"].replace("dp_adpmeta", "dp_fixed")) & (vdf["seed"] == c["seed"])]
        if len(f):
            f = f.iloc[0]
            pairs.append({"run": c["run"], "seed": c["seed"], "ADP_F1": c["MacroF1"], "fixed_F1": f["MacroF1"],
                          "dF1": c["MacroF1"] - f["MacroF1"], "ADP_FAR": c["FAR"], "fixed_FAR": f["FAR"]})
    if pairs:
        pdf_ = pd.DataFrame(pairs)
        print("\nADP-Meta vs fixed noise, paired by seed (validation)\n", pdf_.round(2).to_string(index=False))
        g = pdf_.groupby("run")["dF1"].agg(["mean", "std", "count", lambda x: int((x > 0).sum())])
        g.columns = ["mean_dF1", "std_dF1", "seeds", "seeds_ADP_better"]
        print(g.round(2).to_string())

    agg = vdf.groupby("run")[["MacroF1", "Acc", "FAR", "DR"]].agg(["mean", "std", "count"]).round(2)
    print("\nVALIDATION - mean / std / n over seeds\n", agg.to_string())
    vdf.to_csv(os.path.join(CFG["RESULTS_DIR"], "val_summary.csv"), index=False)
    import zipfile
    with zipfile.ZipFile("results_jsons.zip", "w", zipfile.ZIP_DEFLATED) as z:      # JSON + CSV only (no .pt)
        for fn in os.listdir(CFG["RESULTS_DIR"]):
            if fn.endswith((".json", ".csv")):
                z.write(os.path.join(CFG["RESULTS_DIR"], fn), fn)
    print("Download results_jsons.zip from the Output panel (JSON + CSV only).")


In [ ]:
# BLOCK 18d (NEW - comparison): Torre et al. (TOSEM 2025) 1D-CNN detector inside the Chapter 6 pipeline
# Purpose: compare DETECTORS under identical conditions. Everything except the detector and its loss is the
# Chapter 6 protocol: same leakage-free data (Blocks 5-8b), same 15 Dirichlet clients (alpha 2, run seed),
# same 20 rounds x 400 local DP-SGD steps, batch 256, Adam lr 1e-3, FedProx mu (CFG), per-sample clipping
# with C = CLIP_MULT x median per-sample grad norm on VALIDATION (rule of dp_clip_norm, applied to this model),
# Gaussian noise calibrated with the RDP accountant to the same (eps, delta), same pre-round budget check,
# same model selection (max val macro-F1 with val FAR <= FAR_CAP) and the test set used once (Block 19).
# Detector = Torre et al. Algorithm 1 as printed: Conv1D(32, k=3, ReLU) -> MaxPool1D(2) -> Flatten ->
#   Dense(128, ReLU) -> Dense(softmax). Dropout/BatchNorm are created but not connected in Algorithm 1; they are
#   left out here because BatchNorm mixes samples in a batch and is incompatible with per-sample DP-SGD.
#   Keras initialisers (Glorot-uniform weights, zero biases). Loss = cross-entropy (Algorithm 1, unweighted by
#   default; CNN_CE_WEIGHTS = True uses the same sqrt-balanced weights as the DDQN reward, as an ablation).
# Runs are tagged "torre_cnn" (CFG TAG), so their result files never collide with the DDQN runs.
CNN_STAGE = "cnn_final"                 # None = define only | "smoke" | "cnn_final" (fedavg/fedprox + fixed DP eps 8/3/1, seeds 0/1/2)
CNN_CE_WEIGHTS = False            # ablation: class-weighted cross-entropy (same weights as the DDQN reward)


class TorreCNN(nn.Module):
    def __init__(self, input_dim, n_classes, filters=32, kernel=3, pool=2, dense=128):
        super().__init__()
        self.conv = nn.Conv1d(1, filters, kernel)                                   # L2
        self.pool = nn.MaxPool1d(pool)                                              # L3
        flat = filters * ((input_dim - kernel + 1) // pool)                         # L4
        self.dense = nn.Linear(flat, dense)                                         # L5
        self.out = nn.Linear(dense, n_classes)                                      # L8 (softmax inside the loss)
        for m in (self.conv, self.dense, self.out):
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)               # Keras defaults

    def forward(self, x, mode="q_value"):                                           # x: [B, features] (L1)
        h = self.pool(F.relu(self.conv(x.unsqueeze(1))))
        return self.out(F.relu(self.dense(h.flatten(1))))                           # logits


class CNNAgent:
    """Same interface as SmallDDQN where the federation code needs it; trained by cross-entropy, not reward."""
    def __init__(self, input_dim, action_dim, device, class_weights=None, lr=1e-3, **_):
        self.device, self.action_dim, self.steps = device, action_dim, 0
        self.online_net = TorreCNN(input_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        w = class_weights if (CNN_CE_WEIGHTS and class_weights is not None) else np.ones(action_dim, np.float32)
        self.ce_w = torch.as_tensor(w, dtype=torch.float32, device=device)
        self.eps_start = self.eps_end = 0.0; self.eps_decay_steps = 1                # unused (no exploration)

    def update_target(self):
        pass                                                                        # no target network

    @torch.no_grad()
    def predict(self, x):
        self.online_net.eval()
        out = self.online_net(x.to(self.device)).argmax(1)
        self.online_net.train()
        return out

    def get_fl_parameters(self):
        return [p.detach().cpu().clone() for p in self.online_net.parameters()]

    @torch.no_grad()
    def set_fl_parameters(self, params):
        for p, new in zip(self.online_net.parameters(), params):
            p.data.copy_(new.to(p.device))

    @torch.no_grad()
    def param_vector(self):
        return torch.cat([p.detach().view(-1) for p in self.online_net.parameters()])


def _ce_one(net, w):
    """Loss of ONE record (needed for per-sample clipping): weighted cross-entropy on the logits."""
    def loss_one(p, b, xi, yi):                       # gather (not w[yi]) so that vmap can batch it
        logits = functional_call(net, (p, b), (xi.unsqueeze(0),)).squeeze(0)
        t = yi.unsqueeze(0)
        return -(w.gather(0, t) * F.log_softmax(logits, 0).gather(0, t)).squeeze(0)
    return loss_one


class SupervisedDPClient(DPClient):
    """DPClient._dp_step with the cross-entropy loss; sampling, clipping, noise, accounting, FedProx unchanged."""
    def _dp_step(self, sigma):
        sigma = float(sigma)
        assert sigma >= 0.0, "noise multiplier must be >= 0"
        if sigma > 0:
            assert self.C is not None, "DP guard: noise without clipping has unbounded sensitivity"
            self.accountant.step(noise_multiplier=sigma, sample_rate=self.q)
        else:
            self.nonprivate_steps += 1
        net = self.ddqn.online_net
        idx = (torch.rand(self.n, device=self.device) < self.q).nonzero().squeeze(1)
        params = {k: v.detach() for k, v in net.named_parameters()}
        buffers = {k: v.detach() for k, v in net.named_buffers()}
        if len(idx) > 0:
            x, y = self.X[idx], self.y[idx]
            net.train()
            g = vmap(grad(_ce_one(net, self.ddqn.ce_w)), in_dims=(None, None, 0, 0))(params, buffers, x, y)
            B = len(idx)
            if self.C is not None:
                norms = torch.sqrt(sum(gi.reshape(B, -1).pow(2).sum(1) for gi in g.values()))
                factor = (self.C / (norms + 1e-12)).clamp(max=1.0)
            else:
                factor = torch.ones(B, device=self.device)
            summed = {k: (gi * factor.view(-1, *([1] * (gi.dim() - 1)))).sum(0) for k, gi in g.items()}
        else:
            summed = {k: torch.zeros_like(v) for k, v in params.items()}
        expected_bs = self.q * self.n
        for k, p in net.named_parameters():
            p.grad = summed[k] / expected_bs
            if sigma > 0:
                p.grad = p.grad + torch.normal(0.0, sigma * self.C, size=p.shape, device=self.device) / expected_bs
            if self.global_params is not None and self.mu > 0:
                p.grad = p.grad + self.mu * (p.detach() - self.global_params[k])
        self.ddqn.optimizer.step()
        self.ddqn.optimizer.zero_grad(set_to_none=True)
        self.ddqn.steps += 1


class SupervisedFastFederation(FastFederation):
    """FastFederation._step with the cross-entropy loss (all clients in parallel; same algorithm per client)."""
    def __init__(self, clients):
        super().__init__(clients)
        self._g_ce = vmap(vmap(grad(_ce_one(self.base, clients[0].ddqn.ce_w)), in_dims=(None, None, 0, 0)),
                          in_dims=(0, 0, 0, 0))

    def _step(self, sigma, mu, gp):
        K, dev = self.K, self.dev
        take = (torch.rand(self.row_ok.shape, device=dev) < self.q[:, None]) & self.row_ok
        cnt = take.sum(1)
        B = int(cnt.max())
        if B > 0:
            order = torch.argsort((~take).to(torch.int8), dim=1, stable=True)[:, :B]
            valid = torch.arange(B, device=dev)[None, :] < cnt[:, None]
            x = torch.gather(self.X, 1, order[..., None].expand(-1, -1, self.X.shape[2]))
            y = torch.gather(self.Y, 1, order)
            p_det = {k: v.detach() for k, v in self.params.items()}
            g = self._g_ce(p_det, self.buffers, x, y)                                 # [K, B, ...]
            if self.C is not None:
                norms = torch.sqrt(sum(gi.reshape(K, B, -1).pow(2).sum(-1) for gi in g.values()))
                factor = (self.C / (norms + 1e-12)).clamp(max=1.0)
            else:
                factor = torch.ones(K, B, device=dev)
            factor = factor * valid
            summed = {k: (gi * factor.view(K, B, *([1] * (gi.dim() - 2)))).sum(1) for k, gi in g.items()}
        else:
            summed = {k: torch.zeros_like(v) for k, v in self.params.items()}
        ebs = self.q * self.n
        for k in self.names:
            p = self.params[k]
            shape = (K,) + (1,) * (p.dim() - 1)
            gk = summed[k]
            if bool((sigma > 0).any()):
                gk = gk + torch.randn_like(p) * (sigma * self.C).view(shape)
            gk = gk / ebs.view(shape)
            if mu > 0:
                gk = gk + mu * (p.detach() - gp[k])
            p.grad = gk
        self.opt.step()
        self.opt.zero_grad(set_to_none=True)
        self.steps += 1


ARCH["torre_cnn"] = lambda i, a, d, cw=None, **kw: CNNAgent(i, a, d, cw, **kw)       # Block 19 rebuilds models


def cnn_clip_norm():
    """Same rule as dp_clip_norm: CLIP_MULT x median per-sample grad norm at a seed-42 init, VALIDATION data."""
    set_seed(SPLIT_SEED)
    ag = CNNAgent(input_dim, action_dim, device, reward_weights(), lr=CFG["LR"])
    net = ag.online_net; net.train()
    idx = np.random.default_rng(SPLIT_SEED).choice(len(y_val), size=min(1024, len(y_val)), replace=False)
    x = torch.as_tensor(X_val[idx], device=device); yy = torch.as_tensor(y_val[idx], device=device)
    params = {k: v.detach() for k, v in net.named_parameters()}
    buffers = {k: v.detach() for k, v in net.named_buffers()}
    g = vmap(grad(_ce_one(net, ag.ce_w)), in_dims=(None, None, 0, 0))(params, buffers, x, yy)
    norms = torch.sqrt(sum(gi.reshape(len(idx), -1).pow(2).sum(1) for gi in g.values())).cpu().numpy()
    return CFG["CLIP_MULT"] * float(np.median(norms))


def run_cnn_pipeline(method, seed, eps_budget=None, C=None):
    """run_dp_pipeline (fixed noise / non-private paths) with the CNN detector."""
    mu, use_clip, private, adaptive = METHODS_DP_PIPELINE[method]
    assert not adaptive, "ADP-Meta is not part of this comparison"
    mu = CFG["MU"] if mu == "MU" else mu
    assert private == (eps_budget is not None), f"{method}: eps_budget must be given iff the method is private"
    name = run_name(method, eps_budget)
    R, STEPS, delta = CFG["ROUNDS"], CFG["LOCAL_STEPS"], CFG["DELTA"]
    set_seed(seed)
    rng = np.random.default_rng(seed)
    parts = dirichlet_split(y_dp, NUM_CLIENTS, ALPHA, np.random.default_rng(seed), MIN_PER_CLIENT)
    _a = np.concatenate(parts)
    assert len(_a) == len(y_dp) and len(np.unique(_a)) == len(y_dp), "A record is in more than one client"
    w = reward_weights()
    mk = lambda: CNNAgent(input_dim, action_dim, device, w, lr=CFG["LR"])
    g = mk()
    clients = [SupervisedDPClient(mk(), X_dp[ci], y_dp[ci], device, model_name=f"Client{k+1}",
                                  batch_size=CFG["BATCH"], clip_norm=(C if use_clip else None), delta=delta)
               for k, ci in enumerate(parts)]
    base = ([get_noise_multiplier(target_epsilon=eps_budget, target_delta=delta, sample_rate=c.q,
                                  steps=R * STEPS, accountant="rdp") for c in clients] if private
            else [0.0] * len(clients))
    fast = SupervisedFastFederation(clients) if CFG["FAST"] and CFG["CLIENT_FRAC"] >= 1.0 else None
    print(f"[{name} | seed {seed}] Torre CNN ({sum(p.numel() for p in g.online_net.parameters())} params) | "
          f"clip={C if use_clip else 'none'} | mu={mu} | sigma median={np.median(base):.3f}")
    history, states, eps_now, t0 = [], [], None, time.time()
    for r in range(1, R + 1):
        active = rng.choice(len(clients), size=max(1, math.ceil(CFG["CLIENT_FRAC"] * len(clients))), replace=False)
        if private and max(clients[i].projected_epsilon(base[i], STEPS) for i in active) > eps_budget:
            print(f"[{name}] Round {r}: privacy budget would be exceeded - stopping before spending."); break
        gp = g.get_fl_parameters()
        params, sizes, _ = fit_clients(clients, active, gp, {int(i): base[i] for i in active}, STEPS, mu, fast)
        g.set_fl_parameters(weighted_average(params, sizes))
        h = eval_metrics(g, val_dataset)
        eps_now = max(c.epsilon_spent() for c in clients) if private else None
        if private:
            assert eps_now <= eps_budget + 1e-9, "DP guard: budget exceeded"
        h.update({"round": r, "noise_mult": 1.0, "forced": False, "epsilon": eps_now,
                  "median_sigma": float(np.median(base))})
        history.append(h); states.append([t.clone() for t in g.get_fl_parameters()])
        _log_round(name, r, h, f" | eps={eps_now if eps_now is None else round(eps_now, 3)}")
    return _save(name, seed, history, states, "torre_cnn", {
        "method": method, "private": private, "eps_budget": eps_budget, "final_epsilon": eps_now,
        "delta": delta if private else None, "clip_norm": C if use_clip else None, "mu": mu,
        "base_sigma_per_client": base, "rounds_run": len(history), "minutes": (time.time() - t0) / 60,
        "detector": "Torre et al. 2025 Algorithm 1 1D-CNN", "ce_weights": CNN_CE_WEIGHTS})


def run_cnn_all(experiments, seeds):
    """Like run_all, for the CNN detector. Finished runs are skipped; a changed config under the same tag fails."""
    C = None
    for seed in seeds:
        for method, eps in experiments:
            name = run_name(method, eps)
            if os.path.exists(result_path(name, seed)):
                with open(result_path(name, seed)) as f:
                    old = json.load(f)["cfg"]
                diff = {k for k in set(old) | set(cfg_json()) if k != "RESULTS_DIR" and old.get(k) != cfg_json().get(k)}
                if diff:
                    raise ValueError(f"{name} seed {seed} exists with a DIFFERENT config ({sorted(diff)}).")
                print(f">>> SKIP {name} seed {seed} (done)"); continue
            if METHODS_DP_PIPELINE[method][1] and C is None:
                C = cnn_clip_norm(); print(f"CNN clip norm C (rule of dp_clip_norm, validation): {C:.3f}")
            print(f"\n{'='*70}\n>>> START {name} | seed {seed}\n{'='*70}")
            run_cnn_pipeline(method, seed, eps, C)
            gc.collect()


# Same frozen settings as the DDQN final stage (FINAL_CFG without the ADP-Meta and reward-only keys, which the
# CNN does not use): CLIP_MULT 0.5. REWARD_FA is a DDQN reward term and has no counterpart in cross-entropy.
CNN_CFG = dict(CLIP_MULT=0.5, TAG="torre_cnn" + ("_cew" if CNN_CE_WEIGHTS else ""))
CNN_EXPS = [("fedavg_np", None), ("fedprox_np", None), ("dp_fixed", 8.0), ("dp_fixed", 3.0), ("dp_fixed", 1.0)]
CNN_STAGES = {
    "smoke": (dict(ROUNDS=2, LOCAL_STEPS=5), [0], [("fedprox_np", None), ("dp_fixed", 8.0)]),
    "cnn_final": ({}, [0, 1, 2], CNN_EXPS),
}
if CNN_STAGE is not None:
    over, seeds, exps = CNN_STAGES[CNN_STAGE]
    CFG.clear(); CFG.update(_DEFAULT_CFG); CFG.update(CNN_CFG); CFG.update(over)
    if CNN_STAGE == "smoke":
        CFG["TAG"] += "_smoke"
    run_cnn_all(exps, seeds)
    CFG.clear(); CFG.update(_DEFAULT_CFG)


In [ ]:
# BLOCK 19 (v4 - C0): FINAL TEST - run ONCE, only after every ablation is frozen and the "final" stage is done.
# Evaluates each run's validation-selected model on the test set and reports mean +/- std over seeds.
# Nothing in this cell feeds back into any training or selection decision.
FINAL_TEST = True          # <- set True only for the single final evaluation
TEST_TAG = "torre_cnn"            # only runs of this stage tag are evaluated ("final" for the v2 final stage)

if not FINAL_TEST:
    print("FINAL_TEST is False - test set not touched.")
else:
    rows = []
    for fn in sorted(os.listdir(CFG["RESULTS_DIR"])):
        if not (fn.endswith(".json") and "__seed" in fn) or fn.endswith("_TEST.json"):
            continue
        if TEST_TAG and not fn.split("__seed")[0].endswith(f"__{TEST_TAG}"):
            continue
        with open(os.path.join(CFG["RESULTS_DIR"], fn)) as f:
            r = json.load(f)
        agent = ARCH[r["arch"]](input_dim, action_dim, device, class_weights_dp)
        agent.set_fl_parameters(torch.load(result_path(r["run"], r["seed"], "_best.pt")))
        t = eval_metrics(agent, test_dataset)
        with open(result_path(r["run"], r["seed"], "_TEST.json"), "w") as f:
            json.dump({"run": r["run"], "seed": r["seed"], "best_round": r["best_round"], "cap_met": r["cap_met"],
                       "final_epsilon": r.get("final_epsilon"), "test": t}, f, indent=2)
        rows.append({"run": r["run"], "seed": r["seed"], "cap_met": r["cap_met"], "eps": r.get("final_epsilon"),
                     "Acc": 100 * t["Accuracy"], "MacroF1": 100 * t["Macro_F1"], "WeightedF1": 100 * t["Weighted_F1"],
                     "FAR": 100 * t["FAR"], "DR": 100 * t["DR"],
                     **{f"F1_{k}": 100 * x for k, x in t["F1_per_class"].items()}})
    tdf = pd.DataFrame(rows)
    tdf["eps"] = pd.to_numeric(tdf["eps"])                        # None (non-private) -> NaN
    tdf.to_csv(os.path.join(CFG["RESULTS_DIR"], "test_all_runs.csv"), index=False)
    def pm(s):
        s = s.dropna()
        if s.empty: return "-"
        return f"{s.mean():.2f} ± {s.std(ddof=1):.2f}" if len(s) > 1 else f"{s.mean():.2f} (n=1)"
    cols = ["Acc", "MacroF1", "WeightedF1", "FAR", "DR", "eps"]
    table = tdf.groupby("run").agg({c: pm for c in cols} | {"seed": "count", "cap_met": "all"})
    print("TEST - mean ± std over seeds (sample std, ddof=1)\n", table.to_string())
    per_class = tdf.groupby("run")[[c for c in tdf.columns if c.startswith("F1_")]].agg(pm)
    print("\nTEST - per-class F1 mean ± std\n", per_class.T.to_string())
    table.to_csv(os.path.join(CFG["RESULTS_DIR"], "test_table_mean_std.csv"))

    # Paired ADP-Meta vs fixed noise on TEST (same eps and seed)
    pr = []
    for _, c in tdf[tdf["run"].str.startswith("dp_adpmeta_eps")].iterrows():
        f = tdf[(tdf["run"] == c["run"].replace("dp_adpmeta", "dp_fixed")) & (tdf["seed"] == c["seed"])]
        if len(f):
            pr.append({"run": c["run"], "seed": c["seed"], "dMacroF1": c["MacroF1"] - f.iloc[0]["MacroF1"],
                       "dAcc": c["Acc"] - f.iloc[0]["Acc"], "dFAR": c["FAR"] - f.iloc[0]["FAR"]})
    if pr:
        pdt = pd.DataFrame(pr)
        paired = pdt.groupby("run").agg(dMacroF1=("dMacroF1", pm), dAcc=("dAcc", pm), dFAR=("dFAR", pm),
                                        seeds_ADP_better=("dMacroF1", lambda x: f"{int((x > 0).sum())}/{len(x)}"))
        print("\nTEST - ADP-Meta minus fixed noise, paired by seed (mean ± std)\n", paired.to_string())
        paired.to_csv(os.path.join(CFG["RESULTS_DIR"], "test_paired_adp_vs_fixed.csv"))


In [ ]:
# BLOCK 20 (follow-up): post-processing analysis of the SAVED models - no retraining, no extra privacy cost
#   (a) Binary IDS view (attack vs normal): accuracy, precision, DR, FAR, F1 - shown NEXT TO the 10-class results
#   (b) Per-class ROC-AUC from the DDQN Q-values (ranking quality, independent of the argmax decision)
#   (c) Per-class decision calibration: offsets b added to the Q-values, learned on VALIDATION only
#       (max val macro-F1 with val FAR <= cap). The trained DDQN is unchanged; this is post-processing of a DP
#       model, so the (eps, delta) guarantee is unchanged.
# DISCLOSURE: test numbers produced here are a follow-up, second use of the test set after the main evaluation.
CALIB_TAG = "skip"           # which saved runs to analyse (stage tag)
CALIB_EVAL_TEST = False      # False -> validation only
CALIB_GRID = np.linspace(-1.0, 1.0, 21)   # offsets, in units of the run's Q-value std
CALIB_PASSES = 3

from sklearn.metrics import roc_auc_score


@torch.no_grad()
def q_values(agent, X, bs=4096):
    net = agent.online_net; net.eval()
    out = [net(torch.as_tensor(X[i:i + bs], device=device)).cpu() for i in range(0, len(X), bs)]
    net.train()
    return torch.cat(out).numpy()


def metrics_from_preds(y, p):
    K, n = len(class_names), NORMAL_IDX
    cm = np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    P, R, F1, sup = prf_from_cm(cm)
    att = [i for i in range(K) if i != n]
    tn, fp = cm[n, n], cm[n].sum() - cm[n, n]                       # normal -> normal / normal -> any attack
    fn = cm[att, n].sum(); tp = cm[att].sum() - fn                  # attack -> normal / attack -> any attack
    far = fp / (fp + tn) if fp + tn else 0.0
    dr = tp / (tp + fn) if tp + fn else 0.0
    bprec = tp / (tp + fp) if tp + fp else 0.0
    return {"Acc": np.trace(cm) / cm.sum(), "MacroF1": F1.mean(), "FAR": far, "DR": dr,
            "BinAcc": (tp + tn) / cm.sum(), "BinPrec": bprec,
            "BinF1": 2 * bprec * dr / (bprec + dr) if bprec + dr else 0.0,
            "F1_per_class": dict(zip(class_names, F1))}


def calib_objective(m, cap):
    return m["MacroF1"] - 10.0 * max(0.0, m["FAR"] - cap)


def fit_offsets(Q, y, cap):
    """Coordinate ascent over per-class offsets on VALIDATION. b = 0 is always a candidate, so the objective
    can only stay the same or improve on validation."""
    scale = float(Q.std()) or 1.0
    b = np.zeros(Q.shape[1])
    best = calib_objective(metrics_from_preds(y, Q.argmax(1)), cap)
    for _ in range(CALIB_PASSES):
        for c in range(Q.shape[1]):
            for v in CALIB_GRID * scale:
                trial = b.copy(); trial[c] = v
                obj = calib_objective(metrics_from_preds(y, (Q + trial).argmax(1)), cap)
                if obj > best + 1e-12:
                    best, b = obj, trial
    return b


def per_class_auc(Q, y):
    return {class_names[c]: float(roc_auc_score((y == c).astype(int), Q[:, c]))
            for c in range(Q.shape[1]) if 0 < (y == c).sum() < len(y)}


rows, auc_rows = [], []
print(f"Block 20 | runs tagged '{CALIB_TAG}' | test evaluated: {CALIB_EVAL_TEST}"
      + ("  <- DISCLOSE: follow-up second use of the test set" if CALIB_EVAL_TEST else ""))
for fn in sorted(os.listdir(CFG["RESULTS_DIR"]) if os.path.isdir(CFG["RESULTS_DIR"]) else []):
    if not (fn.endswith(".json") and "__seed" in fn) or fn.endswith("_TEST.json"):
        continue
    if not fn.split("__seed")[0].endswith(f"__{CALIB_TAG}"):
        continue
    with open(os.path.join(CFG["RESULTS_DIR"], fn)) as f:
        r = json.load(f)
    agent = ARCH[r["arch"]](input_dim, action_dim, device, class_weights_dp)
    agent.set_fl_parameters(torch.load(result_path(r["run"], r["seed"], "_best.pt")))
    Qv = q_values(agent, X_val)
    b = fit_offsets(Qv, y_val, CFG["FAR_CAP"])
    splits = [("val", Qv, y_val)] + ([("test", q_values(agent, X_test), y_test)] if CALIB_EVAL_TEST else [])
    for split, Q, y in splits:
        for mode, pred in (("raw", Q.argmax(1)), ("calibrated", (Q + b).argmax(1))):
            m = metrics_from_preds(y, pred)
            rows.append({"run": r["run"], "seed": r["seed"], "split": split, "mode": mode,
                         **{k: 100 * v for k, v in m.items() if k != "F1_per_class"},
                         **{f"F1_{k}": 100 * v for k, v in m["F1_per_class"].items()}})
        auc_rows.append({"run": r["run"], "seed": r["seed"], "split": split, **per_class_auc(Q, y)})

cdf, adf = pd.DataFrame(rows), pd.DataFrame(auc_rows)
if cdf.empty:
    print(f"No saved runs tagged '{CALIB_TAG}' in {CFG['RESULTS_DIR']}")
else:
    def pm(s):
        s = s.dropna()
        return "-" if s.empty else (f"{s.mean():.2f} ± {s.std(ddof=1):.2f}" if len(s) > 1 else f"{s.mean():.2f} (n=1)")
    show = "test" if CALIB_EVAL_TEST else "val"
    raw = cdf[(cdf["split"] == show) & (cdf["mode"] == "raw")]
    print(f"\n(a) BINARY IDS VIEW ({show}, attack vs normal) - report NEXT TO the 10-class results\n",
          raw.groupby("run")[["BinAcc", "BinPrec", "DR", "FAR", "BinF1", "Acc", "MacroF1"]].agg(pm).to_string())
    print(f"\n(b) PER-CLASS ROC-AUC from Q-values ({show}, mean over seeds)\n",
          adf[adf["split"] == show].groupby("run")[list(class_names)].mean().round(3).T.to_string())
    cols = ["Acc", "MacroF1", "FAR", "DR", "BinAcc"]
    for split in (["val", "test"] if CALIB_EVAL_TEST else ["val"]):
        t = cdf[cdf["split"] == split].groupby(["run", "mode"])[cols].agg(pm)
        print(f"\n(c) CALIBRATION ({split}) - offsets learned on validation only\n", t.to_string())
    # paired ADP-Meta minus fixed noise, after calibration
    cal = cdf[(cdf["split"] == show) & (cdf["mode"] == "calibrated")]
    pr = []
    for _, c in cal[cal["run"].str.startswith("dp_adpmeta_eps")].iterrows():
        f = cal[(cal["run"] == c["run"].replace("dp_adpmeta", "dp_fixed")) & (cal["seed"] == c["seed"])]
        if len(f):
            pr.append({"run": c["run"], "seed": c["seed"], "dMacroF1": c["MacroF1"] - f.iloc[0]["MacroF1"],
                       "dAcc": c["Acc"] - f.iloc[0]["Acc"], "dFAR": c["FAR"] - f.iloc[0]["FAR"]})
    if pr:
        pdt = pd.DataFrame(pr)
        print(f"\nADP-Meta minus fixed noise after calibration ({show}, paired by seed)\n",
              pdt.groupby("run").agg(dMacroF1=("dMacroF1", pm), dAcc=("dAcc", pm), dFAR=("dFAR", pm),
                                     seeds_ADP_better=("dMacroF1", lambda x: f"{int((x > 0).sum())}/{len(x)}")).to_string())
    cdf.to_csv(os.path.join(CFG["RESULTS_DIR"], f"postproc_{CALIB_TAG}.csv"), index=False)
    adf.to_csv(os.path.join(CFG["RESULTS_DIR"], f"auc_{CALIB_TAG}.csv"), index=False)
    print(f"\nSaved postproc_{CALIB_TAG}.csv and auc_{CALIB_TAG}.csv in {CFG['RESULTS_DIR']}")
